# City Scan Data Cleaning
##### June 2025

Basic data cleaning pipeline for appropriate CSV preparation necessary for City Scan JavaScript plots with Cartagena, Colombia as the case study example city for pipeline scaling

In [ ]:
# standard library imports
import os
import yaml
import numpy as np
import pandas as pd

os.getcwd()

# # change to project root directory
# os.chdir('../')
# print("directory changes")
# print(f"current working directory is:", os.getpwd())


# local imports (after changing directory)
from Py.clean import clean_pg, clean_pas, clean_uba, clean_uba_area, clean_lc, clean_pug, clean_pv, clean_pv_area, clean_aq_area, clean_ndvi_area, clean_deforestation_area, clean_flood, clean_e, clean_s, clean_ls_area, clean_ee, clean_l_area, clean_fwi



In [155]:
# Load City Directory 
city_dir = open('city-dir.txt').readlines()[0].strip()

user_input_dir = os.path.join(city_dir, '01-user-input/')
process_output_dir = os.path.join(city_dir, '02-process-output/')
spatial_dir = os.path.join(process_output_dir, 'spatial/')
tabular_dir = os.path.join(process_output_dir, 'tabular/')

city_name = city_dir.split('-')[-1].lower() 
country_name = city_dir.split('-')[-2].lower()



if not os.path.exists('source/files.yml'):
    raise FileNotFoundError("The 'source/files.yml' file does not exist. Please create it with the necessary file paths.")

else:
    with open('source/files.yml', 'r') as file:
        files = yaml.safe_load(file)


with open('source/checklist.yml', 'r') as file:
    checks = yaml.safe_load(file)

tabular = [f for f in os.listdir(tabular_dir) if f.endswith('.csv')]
raster = [f for f in os.listdir( spatial_dir) if f.endswith('.tif')]


at = {f.replace(city_name + '_', '').replace('.csv', '') for f in tabular}
ct = set(checks['tabular'])

ar = {f.replace(city_name + '_', '').replace('.tif', '') for f in raster}
cr = set(checks['raster'])

missing_tabular = ct - at

missing_raster = cr - ar

print(missing_tabular, missing_raster)


# if city_name + '_' + 'population-growth.csv' not in os.listdir(tabular_dir):
#     # Get Oxford Population Data
#     get_oxford_pop(city_dir, files, years=[2000, 2021], save=True)

# else:
#     print("Population growth data already exists. Skipping extraction.")

# tabular = [f for f in os.listdir(tabular_dir) if f.endswith('.csv')]
# tifs = [f for f in os.listdir( spatial_dir) if f.endswith('.tif')]



{'earthquake-events', 'monthly-pv'} set()


# POPULATION AND DEMOGRAPHIC TRENDS

### pg.csv preparation
### Observable Notebook functions/charts:
#### 1.) "plot_pga" / "chart_pga" ; and
#### 2.) "plot_pgp" / "chart_pg"

In [14]:
def get_file_by_topic(topic, folder, dir):

    f = [f for f in folder if topic in f.lower()][0]
    filedir = os.path.join(dir, f)

    return filedir

In [16]:
# POPULATION & DEMOGRAPHIC TRENDS - pg.csv preparation for Observable Notebook plot functions/charts:
# 1.) "plot_pga"/"chart_pga" (absolute population growth); and 
# 2.) "plot_pgp"/"chart_pgp" (population growth percentage)

# load "raw" (i.e. "dirty") tabular output data
filename = get_file_by_topic('population-growth', tabular, tabular_dir)
print(filename)

raw_df_pg = pd.read_csv(filename) # updatefile path

# basic info about raw data
print("Raw population growth data info:")
print(f"Shape: {raw_df_pg.shape}")
print(f"Columns: {list(raw_df_pg.columns)}")
print(f"Date range: {raw_df_pg['Year'].min()} - {raw_df_pg['Year'].max()}")
print(f"Data preview:")
print(raw_df_pg.head())
print("\n" + "="*50 + "\n")

# clean data using clean_pg function in clean.py
try:
    cleaned_df_pg = clean_pg(filename) # updatefile path
    print("Population growth data cleaned successfully!")
    
    # cleaned data info
    print(f"\nCleaned data shape: {cleaned_df_pg.shape}")
    print(f"Cleaned data columns: {list(cleaned_df_pg.columns)}")
    print(f"Sample of cleaned data:")
    print(cleaned_df_pg.head(10))
    
    # basic data validation
    print(f"\nData validation:")
    print(f"- Missing values: {cleaned_df_pg.isnull().sum().sum()}")
    print(f"- Year range: {cleaned_df_pg['yearName'].min()} - {cleaned_df_pg['yearName'].max()}")
    print(f"- Population range: {cleaned_df_pg['population'].min():,} - {cleaned_df_pg['population'].max():,}")
    print(f"- Growth rate range: {cleaned_df_pg['populationGrowthPercentage'].min():.3f}% - {cleaned_df_pg['populationGrowthPercentage'].max():.3f}%")
    
    # check for any potential data quality issues
    if cleaned_df_pg['populationGrowthPercentage'].isna().sum() > 0:
        print(f"Note: {cleaned_df_pg['populationGrowthPercentage'].isna().sum()} missing growth rate values (expected for first year)")
    
except Exception as e:
    print(f"Error cleaning population growth data: {e}")
    print("Check that 'data/raw/population-growth.csv' exists and has the correct format")

# save cleaned data as csv file - pg.csv, and export
# (this is handled automatically by clean_pg function, but confirming)
if 'cleaned_df_pg' in locals():
    print(f"Cleaned data saved to: data/processed/pg.csv")
else:
    print("No cleaned data available to save")

mnt/2025-02-tunisia-tunis/02-process-output/tabular/tunis_population-growth.csv
Raw population growth data info:
Shape: (22, 7)
Columns: ['Group', 'Location', 'Country', 'Year', 'Population', 'Source', 'Method']
Date range: 2000 - 2021
Data preview:
   Group Location  Country  Year  Population  Source  Method
0  Tunis    Tunis  Tunisia  2000     1969000  Oxford  Oxford
1  Tunis    Tunis  Tunisia  2001     1984000  Oxford  Oxford
2  Tunis    Tunis  Tunisia  2002     2000000  Oxford  Oxford
3  Tunis    Tunis  Tunisia  2003     2016000  Oxford  Oxford
4  Tunis    Tunis  Tunisia  2004     2035000  Oxford  Oxford


Cleaned data saved to: data/processed/pg.csv
Years covered: 2000 - 2021
Total data points: 22
Population range: 1,969,000 - 2,696,000
Population growth data cleaned successfully!

Cleaned data shape: (22, 3)
Cleaned data columns: ['yearName', 'population', 'populationGrowthPercentage']
Sample of cleaned data:
   yearName  population  populationGrowthPercentage
0      2000     196

### pas.csv preparation
### Observable Notebook functions/charts:
#### 1.) "plot_pas" / "chart_pas"

In [18]:
# POPULATION AGE SEX - pas.csv preparation for Observable Notebook plot functions/charts:
# 1.) "plot_pas"/"chart_pas" (population age sex, i.e., population by sex and age bracket, (i.e., Population Distribution by Age & Sex, xxxx))

# load "raw" (i.e. "dirty") tabular output data
filename = get_file_by_topic('demographics', tabular, tabular_dir)
print(filename)

raw_df_pas = pd.read_csv(filename) # updatefile path

# basic info about raw data
print("Raw population age structure data info:")
print(f"Shape: {raw_df_pas.shape}")
print(f"Columns: {list(raw_df_pas.columns)}")
print(f"Age groups: {sorted(raw_df_pas['age_group'].unique())}")
print(f"Sex categories: {raw_df_pas['sex'].unique()}")
print(f"Total population: {raw_df_pas['population'].sum():,.0f}")
print(f"Data preview:")
print(raw_df_pas.head())
print("\n" + "="*50 + "\n")

# clean data using clean_pas function in clean.py
try:
    cleaned_df_pas = clean_pas(filename) # updatefile path
    print(" Population age structure data cleaned successfully!")
    
    # cleaned data info
    print(f"\nCleaned data shape: {cleaned_df_pas.shape}")
    print(f"Cleaned data columns: {list(cleaned_df_pas.columns)}")
    print(f"Sample of cleaned data:")
    print(cleaned_df_pas.head(10))
    
    # basic data validation
    print(f"\nData validation:")
    print(f"- Missing values: {cleaned_df_pas.isnull().sum().sum()}")
    print(f"- Age brackets: {sorted(cleaned_df_pas['ageBracket'].unique())}")
    print(f"- Sex categories: {sorted(cleaned_df_pas['sex'].unique())}")
    print(f"- Population count range: {cleaned_df_pas['count'].min():,.0f} - {cleaned_df_pas['count'].max():,.0f}")
    print(f"- Percentage range: {cleaned_df_pas['percentage'].min():.3f}% - {cleaned_df_pas['percentage'].max():.3f}%")
    print(f"- Year: {cleaned_df_pas['yearName'].iloc[0]}")
    
    # data quality checks
    total_percentage = cleaned_df_pas['percentage'].sum()
    print(f"- Total percentage sum: {total_percentage:.3f}% (should be ~100%)")
    
    if abs(total_percentage - 100) > 0.1:
        print(f"Warning: Percentage sum deviates from 100% by {abs(total_percentage - 100):.3f}%")
    
    # check for balanced sex representation
    sex_counts = cleaned_df_pas.groupby('sex')['count'].sum()
    print(f"- Population by sex: Female: {sex_counts.get('female', 0):,.0f}, Male: {sex_counts.get('male', 0):,.0f}")
    
    # check age bracket coverage
    expected_brackets = len(cleaned_df_pas['ageBracket'].unique())
    actual_records = len(cleaned_df_pas)
    print(f"- Age brackets: {expected_brackets}, Total records: {actual_records}")
    
    if actual_records != expected_brackets * 2:  # should be 2 records per age bracket (male/female)
        print(f"Note: Expected {expected_brackets * 2} records (2 per age bracket), found {actual_records}")
    
except Exception as e:
    print(f"Error cleaning population age structure data: {e}")
    print("Check that the demographics CSV file exists and has the correct format")
    print("Expected columns: age_group, sex, population")

# save cleaned data as csv file - pas.csv, and export
# (this is handled automatically by clean_pas function, but confirming)
if 'cleaned_df_pas' in locals():
    print(f"Cleaned data saved to: data/processed/pas.csv")
    
    # preview data structure
    print(f"\nData structure summary:")
    print(f"- Columns: {list(cleaned_df_pas.columns)}")
    print(f"- Records per sex: {len(cleaned_df_pas[cleaned_df_pas['sex'] == 'female'])}, {len(cleaned_df_pas[cleaned_df_pas['sex'] == 'male'])}")
    print(f"- Data types: {dict(cleaned_df_pas.dtypes)}")
else:
    print("No cleaned data available to save")

mnt/2025-02-tunisia-tunis/02-process-output/tabular/tunis_demographics.csv
Raw population age structure data info:
Shape: (36, 3)
Columns: ['age_group', 'sex', 'population']
Age groups: ['0-1', '1-4', '10-14', '15-19', '20-24', '25-29', '30-34', '35-39', '40-44', '45-49', '5-9', '50-54', '55-59', '60-64', '65-69', '70-74', '75-79', '80+']
Sex categories: ['f' 'm']
Total population: 2,257,828
Data preview:
  age_group sex    population
0       1-4   f  69171.340703
1       1-4   m  74438.938873
2       0-1   f  15084.418809
3       0-1   m  16249.312061
4       5-9   f  83410.267099


Cleaned data saved to: data/processed/pas.csv
Total population: 2,257,828
Age brackets: 17
Sex categories: 2
Total records: 34
 Population age structure data cleaned successfully!

Cleaned data shape: (34, 5)
Cleaned data columns: ['ageBracket', 'sex', 'count', 'percentage', 'yearName']
Sample of cleaned data:
  ageBracket     sex     count  percentage  yearName
0        0-4  female  84255.76    3.731717  

# BUILT FORM

### URBAN EXTENT AND CHANGE

### uba.csv preparation
### Observable Notebook functions/charts:
#### 1.) "plot_ubaa" / "chart_ubaa" ; and
#### 2.) "plot_ubap" / "chart_ubap"

In [22]:
# URBAN EXTENT AND CHANGE - uba.csv preparation for Observable Notebook plot functions/charts:
# 1.) "plot_ubaa"/"chart_ubaa" (absolute urban extent and change)
# 2.) "plot_ubap"/"chart_ubap" (urban extent and change growth percentage)

# load "raw" (i.e. "dirty") tabular output data

# load "raw" (i.e. "dirty") tabular output data
filename = get_file_by_topic('wsf_stats', tabular, tabular_dir)
print(filename)


raw_df_uba = pd.read_csv(filename) # updatefile path

# basic info about raw data
print("Raw urban built area data info:")
print(f"Shape: {raw_df_uba.shape}")
print(f"Columns: {list(raw_df_uba.columns)}")
print(f"Year range: {raw_df_uba['year'].min()} - {raw_df_uba['year'].max()}")
print(f"UBA range: {raw_df_uba['cumulative sq km'].min():.2f} - {raw_df_uba['cumulative sq km'].max():.2f} sq km")
print(f"Total data points: {len(raw_df_uba)}")
print(f"Data preview:")
print(raw_df_uba.head())
print("\n" + "="*50 + "\n")

# clean data using clean_uba function in clean.py
try:
    cleaned_df_uba = clean_uba(filename) # updatefile path
    print("Urban built area data cleaned successfully!")
    
    # cleaned data info
    print(f"\nCleaned data shape: {cleaned_df_uba.shape}")
    print(f"Cleaned data columns: {list(cleaned_df_uba.columns)}")
    print(f"Sample of cleaned data:")
    print(cleaned_df_uba.head(10))
    
    # basic data validation
    print(f"\nData validation:")
    print(f"- Missing values: {cleaned_df_uba.isnull().sum().sum()}")
    print(f"- Year range: {cleaned_df_uba['yearName'].min()} - {cleaned_df_uba['yearName'].max()}")
    print(f"- UBA range: {cleaned_df_uba['uba'].min():.2f} - {cleaned_df_uba['uba'].max():.2f} sq km")
    print(f"- Growth rate range: {cleaned_df_uba['ubaGrowthPercentage'].min():.3f}% - {cleaned_df_uba['ubaGrowthPercentage'].max():.3f}%")
    print(f"- Total urban expansion: {cleaned_df_uba['uba'].max() - cleaned_df_uba['uba'].min():.2f} sq km over {cleaned_df_uba['yearName'].max() - cleaned_df_uba['yearName'].min()} years")
    
    # data quality checks
    print(f"\nUrban growth analysis:")
    # calculate average annual growth rate
    avg_growth = cleaned_df_uba['ubaGrowthPercentage'].mean()
    print(f"- Average annual UBA growth rate: {avg_growth:.3f}%")
    
    # check for any potential data quality issues
    if cleaned_df_uba['ubaGrowthPercentage'].isna().sum() > 0:
        print(f"Note: {cleaned_df_uba['ubaGrowthPercentage'].isna().sum()} missing growth rate values (expected for first year)")
    
    # check for negative growth (Note: urban area should generally increase)
    negative_growth = cleaned_df_uba[cleaned_df_uba['ubaGrowthPercentage'] < 0]
    if len(negative_growth) > 0:
        print(f"Warning: {len(negative_growth)} years with negative UBA growth detected")
        print(f"Years with decline: {negative_growth['yearName'].tolist()}")
      
except Exception as e:
    print(f"Error cleaning urban built area data: {e}")
    print("Check that the UBA CSV file exists and has the correct format")
    print("Expected columns: year, cumulative sq km")

# save cleaned data as csv file - uba.csv, and export
# (this is handled automatically by clean_uba function, but confirming)
if 'cleaned_df_uba' in locals():
    print(f"\nCleaned data saved to: data/processed/uba.csv")
    
    # preview data structure
    print(f"\nData structure summary:")
    print(f"- Columns: {list(cleaned_df_uba.columns)}")
    print(f"- Time series length: {len(cleaned_df_uba)} years")
    print(f"- Data types: {dict(cleaned_df_uba.dtypes)}")
else:
    print("No cleaned data available to save")

mnt/2025-02-tunisia-tunis/02-process-output/tabular/tunis_wsf_stats.csv
Raw urban built area data info:
Shape: (31, 2)
Columns: ['year', 'cumulative sq km']
Year range: 1985 - 2015
UBA range: 151.36 - 265.65 sq km
Total data points: 31
Data preview:
   year  cumulative sq km
0  1985        151.358738
1  1986        158.841053
2  1987        167.717644
3  1988        176.013653
4  1989        181.730767


Cleaned data saved to: data/processed/uba.csv
Years covered: 1985 - 2015
Total data points: 31
UBA range: 151.36 - 265.65 sq km
Urban built area data cleaned successfully!

Cleaned data shape: (31, 4)
Cleaned data columns: ['year', 'yearName', 'uba', 'ubaGrowthPercentage']
Sample of cleaned data:
   year  yearName     uba  ubaGrowthPercentage
0     1      1985  151.36                  NaN
1     2      1986  158.84                4.942
2     3      1987  167.72                5.591
3     4      1988  176.01                4.943
4     5      1989  181.73                3.250
5     6     

### uba_area.csv preparation (i.e., % area with different years of built-up area urban expansion - "Before 1985","1986-1995","1996-2005", and "2006-2015")
### Observable Notebook functions/charts:
#### 1.) "plot_uba_area" / "chart_uba_area" (i.e., Percentage of Area with different years of urban built-up area expansion, "Before 1985", "1986-1995", "1996-2005", "2006-2015")


In [24]:
# URBAN EXTENT AND CHANGE - uba_area.csv preparation from raw tif data for Observable Notebook plot functions/charts:
# 1.) "plot_uba_area"/"chart_uba_area" (i.e., Percentage of Area with different years of urban built-up area expansion, "Before 1985", "1986-1995", "1996-2005", and "2006-2015")

# load "raw" (i.e. "dirty") tif data
input_tif_path = get_file_by_topic('wsf_evolution_utm', raster, spatial_dir)
print(input_tif_path)

# input_tif_path = 'data/raw/cartagena_main_wsf_evolution_utm.tif'  # updatefile path

print("="*60)
print("URBAN BUILT-UP AREA EXPANSION DATA PROCESSING")
print("="*60)

# process tif file using clean_uba_area function
try:
    cleaned_df_uba = clean_uba_area(input_tif_path)
    print("Urban expansion data processed successfully!")
    
    # cleaned data structure
    print(f"\nCleaned data shape: {cleaned_df_uba.shape}")
    print(f"Cleaned data columns: {list(cleaned_df_uba.columns)}")
    print(f"\nProcessed Urban Expansion data:")
    print(cleaned_df_uba)
    
    # basic data validation
    print(f"\nData Validation:")
    print(f"- Missing values: {cleaned_df_uba.isnull().sum().sum()}")
    print(f"- Urban expansion periods: {len(cleaned_df_uba)}")
    print(f"- Total pixels: {cleaned_df_uba['count'].sum():,.0f}")
    print(f"- Percentage sum: {cleaned_df_uba['percentage'].sum():.1f}% (should be ~100%)")
    
    # urban expansion temporal analysis
    print(f"\nUrban Expansion Timeline:")
    
    total_pixels = cleaned_df_uba['count'].sum()
    
    for idx, row in cleaned_df_uba.iterrows():
        print(f"- {row['bin']}: {row['count']:,.0f} pixels ({row['percentage']:.1f}%)")
    
    # ID most and least active expansion periods
    if len(cleaned_df_uba) > 0:
        # filter out zero-count periods for meaningful analysis
        active_periods = cleaned_df_uba[cleaned_df_uba['count'] > 0]
        
        if len(active_periods) > 0:
            max_expansion = active_periods.loc[active_periods['percentage'].idxmax()]
            min_expansion = active_periods.loc[active_periods['percentage'].idxmin()]
            
            print(f"\n- Most active expansion period: {max_expansion['bin']} ({max_expansion['percentage']:.1f}%)")
            print(f"- Least active expansion period: {min_expansion['bin']} ({min_expansion['percentage']:.1f}%)")
    
    # historical vs recent development
    before_2000 = cleaned_df_uba[cleaned_df_uba['bin'].isin(['Before 1985', '1986-1995'])]['percentage'].sum()
    after_2000 = cleaned_df_uba[cleaned_df_uba['bin'].isin(['1996-2005', '2006-2015'])]['percentage'].sum()
    
    print(f"\nTemporal Distribution:")
    print(f"- Historical development (before 1996): {before_2000:.1f}%")
    print(f"- Recent development (1996-2015): {after_2000:.1f}%")
    
    # data quality checks
    print(f"\nData Quality Checks:")
    
    quality_issues = 0
    
    # check for missing values
    missing_values = cleaned_df_uba.isnull().sum().sum()
    if missing_values > 0:
        print(f"Missing values detected: {missing_values}")
        quality_issues += 1
    
    # check for negative values (should not exist for counts)
    negative_counts = (cleaned_df_uba['count'] < 0).sum()
    negative_percentages = (cleaned_df_uba['percentage'] < 0).sum()
    
    if negative_counts > 0:
        print(f"Negative count values: {negative_counts}")
        quality_issues += 1
    if negative_percentages > 0:
        print(f"Negative percentage values: {negative_percentages}")
        quality_issues += 1
    
    # check percentage sum
    percentage_sum = cleaned_df_uba['percentage'].sum()
    if abs(percentage_sum - 100) > 0.1:
        print(f"Percentage sum deviation: {percentage_sum:.1f}% (should be ~100%)")
        quality_issues += 1
    
    # check for duplicate periods
    duplicates = cleaned_df_uba['bin'].duplicated().sum()
    if duplicates > 0:
        print(f"Duplicate time periods: {duplicates}")
        quality_issues += 1
    
    # check for expected number of periods (should be 4)
    if len(cleaned_df_uba) != 4:
        print(f"Unexpected number of periods: {len(cleaned_df_uba)} (expected 4)")
        quality_issues += 1
    
    if quality_issues == 0:
        print("No data quality issues detected")
    
    # data structure summary
    print(f"\nData structure summary:")
    print(f"- Columns: {list(cleaned_df_uba.columns)}")
    print(f"- Time periods: {len(cleaned_df_uba)} expansion periods")
    print(f"- Data types: {dict(cleaned_df_uba.dtypes)}")
    
    print(f"\nOutput file ready: data/processed/uba_area.csv")

    
except Exception as e:
    print(f"Error processing urban expansion data: {e}")
    print("Troubleshooting steps:")
    print("   1. Ensure TIF file exists at the specified path")
    print("   2. Check that the TIF file contains valid year data (1900-2030 range)")
    print("   3. Verify the TIF file is not corrupted")
    print("   4. Ensure rasterio library is installed: pip install rasterio")
    print("   5. Check file permissions and disk space")
    print("   6. Verify year values in TIF are realistic (e.g., 1985, 2005, etc.)")

print("\n" + "="*60)

mnt/2025-02-tunisia-tunis/02-process-output/spatial/tunis_wsf_evolution_utm.tif
URBAN BUILT-UP AREA EXPANSION DATA PROCESSING
Cleaned UBA data saved to: data/processed/uba_area.csv
Urban expansion periods: 4
Total pixels analyzed: 362,380
Percentage coverage verification: 100.0% (should be ~100%)
Dominant expansion period: Before 1985 (57.0%)
Year range in data: 1985 - 2015
Urban expansion data processed successfully!

Cleaned data shape: (4, 4)
Cleaned data columns: ['bin', 'year', 'count', 'percentage']

Processed Urban Expansion data:
           bin       year   count  percentage
0  Before 1985      ≤1985  206476       56.98
1    1986-1995  1986-1995   80482       22.21
2    1996-2005  1996-2005   44884       12.39
3    2006-2015  2006-2015   30538        8.43

Data Validation:
- Missing values: 0
- Urban expansion periods: 4
- Total pixels: 362,380
- Percentage sum: 100.0% (should be ~100%)

Urban Expansion Timeline:
- Before 1985: 206,476 pixels (57.0%)
- 1986-1995: 80,482 pixels 

### LAND COVER

### lc.csv preparation
### Observable Notebook functions/charts:
#### 1.) "plot_lc" / "chart_lc

In [26]:
# LAND COVER - lc.csv preparation for Observable Notebook plot functions/charts:
# 1.) "plot_lc"/"chart_lc" (land cover types)

# load "raw" (i.e. "dirty") tabular output data

filename = get_file_by_topic('lc', tabular, tabular_dir)
print(filename)

raw_df_lc = pd.read_csv(filename) # updatefile path

# basic info about raw data
print("Raw land cover data info:")
print(f"Shape: {raw_df_lc.shape}")
print(f"Columns: {list(raw_df_lc.columns)}")
print(f"Total data points: {len(raw_df_lc)}")

# preview land cover types and pixel counts
if 'Land Cover Type' in raw_df_lc.columns and 'Pixel Count' in raw_df_lc.columns:
    print(f"Land cover types: {raw_df_lc['Land Cover Type'].nunique()}")
    print(f"Total pixels: {raw_df_lc['Pixel Count'].sum():,.0f}")
    print(f"Pixel count range: {raw_df_lc['Pixel Count'].min():,.0f} - {raw_df_lc['Pixel Count'].max():,.0f}")

print(f"Data preview:")
print(raw_df_lc.head())
print("\n" + "="*50 + "\n")

# clean data using clean_lc function in clean.py
try:
    cleaned_df_lc = clean_lc(filename) # updatefile path
    print("Land cover data cleaned successfully!")
    
    # cleaned data info
    print(f"\nCleaned data shape: {cleaned_df_lc.shape}")
    print(f"Cleaned data columns: {list(cleaned_df_lc.columns)}")
    print(f"Sample of cleaned data:")
    print(cleaned_df_lc.head(10))
    
    # basic data validation
    print(f"\nData validation:")
    print(f"- Missing values: {cleaned_df_lc.isnull().sum().sum()}")
    print(f"- Land cover types: {len(cleaned_df_lc)}")
    print(f"- Total pixels: {cleaned_df_lc['pixelTotal'].iloc[0]:,.0f}")
    print(f"- Percentage sum: {cleaned_df_lc['percentage'].sum():.1f}% (should be ~100%)")
    
    # land cover analysis
    print(f"\nLand Cover Data Summary:")
    
    print(f"- Land cover types present: {len(cleaned_df_lc)}")
    print(f"- Pixel count range: {cleaned_df_lc['pixelCount'].min():,.0f} - {cleaned_df_lc['pixelCount'].max():,.0f}")
    
    # ID extremes
    dominant_type = cleaned_df_lc.iloc[0]  # first row after sorting by percentage
    least_common_type = cleaned_df_lc.iloc[-1]  # last row after sorting
    
    print(f"- Most common land cover: {dominant_type['lcType']} ({dominant_type['percentage']:.1f}%)")
    print(f"- Least common land cover: {least_common_type['lcType']} ({least_common_type['percentage']:.1f}%)")
    
    # coverage distribution
    above_10_percent = (cleaned_df_lc['percentage'] >= 10).sum()
    above_5_percent = (cleaned_df_lc['percentage'] >= 5).sum()
    above_1_percent = (cleaned_df_lc['percentage'] >= 1).sum()
    
    print(f"- Types with ≥10% coverage: {above_10_percent}")
    print(f"- Types with ≥5% coverage: {above_5_percent}")
    print(f"- Types with ≥1% coverage: {above_1_percent}")
    
    # data quality checks
    print(f"\nData Quality Checks:")
    
    quality_issues = 0
    
    # check for missing values in key columns
    missing_type = cleaned_df_lc['lcType'].isna().sum()
    missing_count = cleaned_df_lc['pixelCount'].isna().sum()
    missing_percentage = cleaned_df_lc['percentage'].isna().sum()
    
    if missing_type > 0:
        print(f"Missing land cover type values: {missing_type}")
        quality_issues += 1
    if missing_count > 0:
        print(f"Missing pixel count values: {missing_count}")
        quality_issues += 1
    if missing_percentage > 0:
        print(f"Missing percentage values: {missing_percentage}")
        quality_issues += 1
    
    # check for impossible values
    negative_pixels = (cleaned_df_lc['pixelCount'] < 0).sum()
    negative_percentage = (cleaned_df_lc['percentage'] < 0).sum()
    
    if negative_pixels > 0:
        print(f"Negative pixel count values: {negative_pixels}")
        quality_issues += 1
    if negative_percentage > 0:
        print(f"Negative percentage values: {negative_percentage}")
        quality_issues += 1
    
    # check percentage sum
    percentage_sum = cleaned_df_lc['percentage'].sum()
    if abs(percentage_sum - 100) > 0.1:
        print(f"Percentage sum deviation: {percentage_sum:.1f}% (should be ~100%)")
        quality_issues += 1
    
    # check for duplicate land cover types
    duplicates = cleaned_df_lc['lcType'].duplicated().sum()
    if duplicates > 0:
        print(f"Duplicate land cover types: {duplicates}")
        quality_issues += 1
    
    if quality_issues == 0:
        print("No data quality issues detected")
        
except Exception as e:
    print(f"Error cleaning land cover data: {e}")
    print("Check that the land cover CSV file exists and has the correct format")
    print("Expected columns: Land Cover Type, Pixel Count")

# save confirmation and next steps
if 'cleaned_df_lc' in locals():
    print(f"\nCleaned data saved to: data/processed/lc.csv")
    
    # preview of data structure
    print(f"\nData structure summary for Observable:")
    print(f"- Columns: {list(cleaned_df_lc.columns)}")
    print(f"- Data points: {len(cleaned_df_lc)} land cover types")
    print(f"- Data types: {dict(cleaned_df_lc.dtypes)}")
    print(f"- Coverage range: {cleaned_df_lc['percentage'].min():.1f}% - {cleaned_df_lc['percentage'].max():.1f}%")
    
else:
    print("No cleaned land cover data available")
    print("Troubleshooting steps:")
    print("   1. Ensure land cover CSV file exists in data/raw/")
    print("   2. Check file has correct columns: Land Cover Type, Pixel Count")
    print("   3. Verify pixel count values are numeric and positive")
    print("   4. Check that land cover types are properly named")

mnt/2025-02-tunisia-tunis/02-process-output/tabular/tunis_lc.csv
Raw land cover data info:
Shape: (11, 2)
Columns: ['Land Cover Type', 'Pixel Count']
Total data points: 11
Land cover types: 11
Total pixels: 6,068,006
Pixel count range: 0 - 3,438,092
Data preview:
  Land Cover Type   Pixel Count
0      Tree cover  3.850351e+05
1       Shrubland  1.515353e+04
2       Grassland  1.022592e+06
3        Cropland  3.467317e+05
4        Built-up  3.438092e+06


Cleaned data saved to: data/processed/lc.csv
Land cover types: 8
Total pixels analyzed: 6,068,006
Percentage coverage verification: 100.0% (should be ~100%)
Dominant land cover: Built-up (56.7%)
Land cover data cleaned successfully!

Cleaned data shape: (8, 4)
Cleaned data columns: ['lcType', 'pixelCount', 'pixelTotal', 'percentage']
Sample of cleaned data:
                     lcType  pixelCount    pixelTotal  percentage
0                  Built-up     3438092  6.068006e+06       56.66
1                 Grassland     1022592  6.068006e

# URBAN DEVELOPMENT DYNAMICS MATRIX

### pug.csv preparation
### Observable Notebook functions/charts:
#### 1.) "plot_uddm" / "chart_uddm"

In [29]:
# URBAN DEVELOPMENT DYNAMICS MATRIX: POPULATION URBAN GROWTH RATIO - pug.csv preparation for Observable Notebook plot functions/charts:
# 1.) "plot_uddm"/"chart_ud" (population vs urban growth analysis)

# prereqs: ensure pg.csv and uba.csv have been generated from clean_pg and clean_uba functions
print("Checking prerequisite files...")

# check if required input files exist
import os
pg_file = 'data/processed/pg.csv'
uba_file = 'data/processed/uba.csv'

if os.path.exists(pg_file):
    print(f"Population growth file found: {pg_file}")
else:
    print(f"Population growth file missing: {pg_file}")
    print("Run clean_pg function first to generate this file")

if os.path.exists(uba_file):
    print(f"Urban built area file found: {uba_file}")
else:
    print(f"Urban built area file missing: {uba_file}")
    print("Run clean_uba function first to generate this file")

print("\n" + "="*50 + "\n")

# clean and merge data using clean_pug function in clean.py
try:
    cleaned_df_pug = clean_pug()  # uses default paths: pg.csv and uba.csv
    print("Population urban growth data merged and cleaned successfully!")
    
    # cleaned data info
    print(f"\nCleaned data shape: {cleaned_df_pug.shape}")
    print(f"Cleaned data columns: {list(cleaned_df_pug.columns)}")
    print(f"Sample of cleaned data:")
    print(cleaned_df_pug.head(10))
    
    # basic data validation
    print(f"\nData validation:")
    print(f"- Missing values: {cleaned_df_pug.isnull().sum().sum()}")
    print(f"- Year range: {cleaned_df_pug['yearName'].min()} - {cleaned_df_pug['yearName'].max()}")
    print(f"- Population range: {cleaned_df_pug['population'].min():,} - {cleaned_df_pug['population'].max():,}")
    print(f"- UBA range: {cleaned_df_pug['uba'].min():.2f} - {cleaned_df_pug['uba'].max():.2f} sq km")
    print(f"- Density range: {cleaned_df_pug['density'].min():.1f} - {cleaned_df_pug['density'].max():.1f} people/sq km")
    
    # population vs urban growth analysis
    print(f"\nPopulation vs Urban Growth Analysis:")
    print(f"- Population growth rate range: {cleaned_df_pug['populationGrowthPercentage'].min():.3f}% - {cleaned_df_pug['populationGrowthPercentage'].max():.3f}%")
    print(f"- UBA growth rate range: {cleaned_df_pug['ubaGrowthPercentage'].min():.3f}% - {cleaned_df_pug['ubaGrowthPercentage'].max():.3f}%")
    
    # calculate averages (excluding "NaN" values)
    avg_pop_growth = cleaned_df_pug['populationGrowthPercentage'].mean()
    avg_uba_growth = cleaned_df_pug['ubaGrowthPercentage'].mean()
    print(f"- Average annual population growth: {avg_pop_growth:.3f}%")
    print(f"- Average annual UBA growth: {avg_uba_growth:.3f}%")
    
    # growth ratio analysis
    valid_ratios = cleaned_df_pug['populationUrbanGrowthRatio'].dropna()
    if len(valid_ratios) > 0:
        print(f"- Population/Urban growth ratio range: {valid_ratios.min():.3f} - {valid_ratios.max():.3f}")
        print(f"- Average growth ratio: {valid_ratios.mean():.3f}")
        
        # interpret growth patterns
        if valid_ratios.mean() > 1:
            print("Population growing faster than urban area (potential densification)")
        elif valid_ratios.mean() < 1:
            print("Urban area growing faster than population (potential sprawl)")
        else:
            print("Balanced population and urban growth")
    
    # density analysis
    print(f"\nUrban Density Analysis:")
    density_change = cleaned_df_pug['density'].iloc[-1] - cleaned_df_pug['density'].iloc[0]
    density_change_pct = (density_change / cleaned_df_pug['density'].iloc[0]) * 100
    print(f"- Starting density ({cleaned_df_pug['yearName'].iloc[0]}): {cleaned_df_pug['density'].iloc[0]:,.1f} people/sq km")
    print(f"- Ending density ({cleaned_df_pug['yearName'].iloc[-1]}): {cleaned_df_pug['density'].iloc[-1]:,.1f} people/sq km")
    print(f"- Total density change: {density_change:+.1f} people/sq km ({density_change_pct:+.1f}%)")
    
    # data quality checks
    print(f"\nData Quality Checks:")
    
    # check for missing ratios
    missing_ratios = cleaned_df_pug['populationUrbanGrowthRatio'].isna().sum()
    if missing_ratios > 0:
        print(f"Note: {missing_ratios} missing growth ratio values (possibly due to zero UBA growth)")
        zero_uba_growth = cleaned_df_pug[cleaned_df_pug['ubaGrowthPercentage'] == 0]
        if len(zero_uba_growth) > 0:
            print(f"Years with zero UBA growth: {zero_uba_growth['yearName'].tolist()}")
    
    # check for negative population growth
    negative_pop_growth = cleaned_df_pug[cleaned_df_pug['populationGrowthPercentage'] < 0]
    if len(negative_pop_growth) > 0:
        print(f"Note: {len(negative_pop_growth)} years with population decline")
        print(f"Decline years: {negative_pop_growth['yearName'].tolist()}")
    
    # check for negative urban growth
    negative_uba_growth = cleaned_df_pug[cleaned_df_pug['ubaGrowthPercentage'] < 0]
    if len(negative_uba_growth) > 0:
        print(f"Warning: {len(negative_uba_growth)} years with urban area decline")
        print(f"UBA decline years: {negative_uba_growth['yearName'].tolist()}")
    
except Exception as e:
    print(f"Error creating population urban growth data: {e}")
    print("Check that both pg.csv and uba.csv files exist and have the correct format")
    print("Expected pg.csv columns: yearName, population, populationGrowthPercentage")
    print("Expected uba.csv columns: yearName, uba, ubaGrowthPercentage")

# save confirmation and next steps
if 'cleaned_df_pug' in locals():
    print(f"\nCleaned data saved to: data/processed/pug.csv")
    
    # preview of data structure
    print(f"\nData structure summary:")
    print(f"- Columns: {list(cleaned_df_pug.columns)}")
    print(f"- Time series length: {len(cleaned_df_pug)} years")
    print(f"- Data types: {dict(cleaned_df_pug.dtypes)}")
    print(f"- Overlapping years between datasets: {len(cleaned_df_pug)} out of potential maximum")
    
    # summary statistics
    print(f"\nSummary statistics:")
    total_pop_growth = ((cleaned_df_pug['population'].iloc[-1] / cleaned_df_pug['population'].iloc[0]) - 1) * 100
    total_uba_growth = ((cleaned_df_pug['uba'].iloc[-1] / cleaned_df_pug['uba'].iloc[0]) - 1) * 100
    print(f"- Total population growth over period: {total_pop_growth:.1f}%")
    print(f"- Total urban area growth over period: {total_uba_growth:.1f}%")
    print(f"- NET population density change: {density_change_pct:+.1f}%")
    
else:
    print("No cleaned data available to save")
    print("Troubleshooting steps:")
    print("   1. Ensure pg.csv exists (run clean_pg function)")
    print("   2. Ensure uba.csv exists (run clean_uba function)")
    print("   3. Check that both files have overlapping years")

Checking prerequisite files...
Population growth file found: data/processed/pg.csv
Urban built area file found: data/processed/uba.csv


Successfully loaded population growth data: 22 records
Successfully loaded urban built area data: 31 records
Successfully merged datasets: 16 overlapping years
Cleaned data saved to: data/processed/pug.csv
Years covered: 2000 - 2015
Total data points: 16
Population range: 1,969,000 - 2,449,000
UBA range: 226.19 - 265.65
Density range: 8446.1 - 9218.9
Note: 1 missing growth ratios (likely due to zero UBA growth)
Population urban growth data merged and cleaned successfully!

Cleaned data shape: (16, 8)
Cleaned data columns: ['yearName', 'population', 'populationGrowthPercentage', 'year', 'uba', 'ubaGrowthPercentage', 'density', 'populationUrbanGrowthRatio']
Sample of cleaned data:
   yearName  population  populationGrowthPercentage  year     uba  \
0      2000     1969000                         NaN    16  226.19   
1      2001     1984000              

# CLIMATE CONDITIONS

### pv.csv preparation (i.e., monthly max pv potential)
### Observable Notebook functions/charts:
#### 1.) "plot_pv" / "chart_pv" (i.e., Seasonal availability of solar energy, January - December)
#### 2.) "plot_pv_alt" / "chart_pv_alt" (i.e., Seasonal availability of solar energy, January - December with colored conditions for "Excellent (4.5+)", "Favorable (3.5-4.5)", and "Less than Favorable (<3.5)" conditions)
#### 3.) "plot_pv_d" / "chart_pv_d" (i.e., Photovoltaic Potenital Condition Level Distribution)

In [33]:
# PHOTOVOLTAIC POTENTIAL - pv.csv preparation for Observable Notebook plot functions/charts:
# 1.) "plot_pv"/"chart_pv" (i.e., Seasonal availability of solar energy, January - December)
# 2.) "plot_pv_alt"/"chart_pv_alt" (i.e., Seasonal availability of solar energy, January - December with colored conditions for "Excellent (4.5+)", "Favorable (3.5-4.5)", and "Less than Favorable (<3.5)" conditions)
# 3.) "plot_pv_d"/"chart_pv_d" (i.e., Photovoltaic Potenital Condition Level Distribution)

# load "raw" (i.e. "dirty") tabular output data

filename = get_file_by_topic('monthly-pv', tabular, tabular_dir)
print(filename)

raw_df_pv = pd.read_csv(filename) # updatefile path



# basic info about raw data
print("Raw photovoltaic potential data info:")
print(f"Shape: {raw_df_pv.shape}")
print(f"Columns: {list(raw_df_pv.columns)}")
print(f"Month range: {raw_df_pv['month'].min()} - {raw_df_pv['month'].max()}")
print(f"PV max range: {raw_df_pv['max'].min():.2f} - {raw_df_pv['max'].max():.2f}")
print(f"PV mean range: {raw_df_pv['mean'].min():.2f} - {raw_df_pv['mean'].max():.2f}")
print(f"Total data points: {len(raw_df_pv)}")
print(f"Data preview:")
print(raw_df_pv.head())
print("\n" + "="*50 + "\n")

# clean data using clean_pv function in clean.py
try:
    cleaned_df_pv = clean_pv(filename) # updatefile path
    print("Photovoltaic potential data cleaned successfully!")
    
    # cleaned data info
    print(f"\nCleaned data shape: {cleaned_df_pv.shape}")
    print(f"Cleaned data columns: {list(cleaned_df_pv.columns)}")
    print(f"Sample of cleaned data:")
    print(cleaned_df_pv.head(12))  # show all 12 months
    
    # basic data validation
    print(f"\nData validation:")
    print(f"- Missing values: {cleaned_df_pv.isnull().sum().sum()}")
    print(f"- Month coverage: {len(cleaned_df_pv)} months (should be 12)")
    print(f"- Month range: {cleaned_df_pv['month'].min()} - {cleaned_df_pv['month'].max()}")
    print(f"- PV potential range: {cleaned_df_pv['maxPv'].min():.2f} - {cleaned_df_pv['maxPv'].max():.2f}")
    
    # solar energy analysis
    print(f"\nSolar Energy Analysis:")
    
    # ID peak and low months
    peak_month = cleaned_df_pv.loc[cleaned_df_pv['maxPv'].idxmax()]
    low_month = cleaned_df_pv.loc[cleaned_df_pv['maxPv'].idxmin()]
    
    print(f"- Peak solar month: {peak_month['monthName']} ({peak_month['maxPv']:.2f})")
    print(f"- Lowest solar month: {low_month['monthName']} ({low_month['maxPv']:.2f})")
    
    # seasonal analysis
    spring_months = cleaned_df_pv[cleaned_df_pv['month'].isin([3, 4, 5])]  # Mar, Apr, May
    summer_months = cleaned_df_pv[cleaned_df_pv['month'].isin([6, 7, 8])]  # Jun, Jul, Aug
    fall_months = cleaned_df_pv[cleaned_df_pv['month'].isin([9, 10, 11])]  # Sep, Oct, Nov
    winter_months = cleaned_df_pv[cleaned_df_pv['month'].isin([12, 1, 2])]  # Dec, Jan, Feb
    
    spring_avg = spring_months['maxPv'].mean()
    summer_avg = summer_months['maxPv'].mean()
    fall_avg = fall_months['maxPv'].mean()
    winter_avg = winter_months['maxPv'].mean()
    
    print(f"- Spring average (Mar-May): {spring_avg:.2f}")
    print(f"- Summer average (Jun-Aug): {summer_avg:.2f}")
    print(f"- Fall average (Sep-Nov): {fall_avg:.2f}")
    print(f"- Winter average (Dec-Feb): {winter_avg:.2f}")
    
    # calculate seasonal variations
    annual_avg = cleaned_df_pv['maxPv'].mean()
    peak_variation = ((cleaned_df_pv['maxPv'].max() - annual_avg) / annual_avg) * 100 # (i.e., "the best month 6% better than the average"
    low_variation = ((annual_avg - cleaned_df_pv['maxPv'].min()) / annual_avg) * 100 # (i.e., "the worst month 10% worse than the average")
    
    print(f"- Annual average: {annual_avg:.2f}")
    print(f"- Peak month deviation: +{peak_variation:.1f}% above average")
    print(f"- Low month deviation: -{low_variation:.1f}% below average")
    
    # energy planning insights
    summer_winter_ratio = summer_avg / winter_avg
    print(f"- Summer/Winter ratio: {summer_winter_ratio:.2f}x")
    
    # data quality checks
    print(f"\nData Quality Checks:")
    
    # check for complete month coverage
    expected_months = set(range(1, 13))
    actual_months = set(cleaned_df_pv['month'].unique())
    missing_months = expected_months - actual_months
    
    if missing_months:
        print(f"Warning: Missing months: {sorted(missing_months)}")
    else:
        print("Complete 12-month coverage")
    
    # check for reasonable PV values
    if cleaned_df_pv['maxPv'].min() < 0:
        print(f"Warning: Negative PV values detected (minimum: {cleaned_df_pv['maxPv'].min():.2f})")
    
    # ID unusual patterns
    monthly_diff = cleaned_df_pv['maxPv'].diff().abs()

    
except Exception as e:
    print(f"Error cleaning photovoltaic potential data: {e}")
    print("Check that the monthly-pv.csv file exists and has the correct format")
    print("Expected columns: month, max, min, mean")

# save cleaned data as csv file - pv.csv, and export
# (this is handled automatically by clean_pv function, but confirming)
if 'cleaned_df_pv' in locals():
    print(f"\nCleaned data saved to: data/processed/pv.csv")
    
    # preview of data structure
    print(f"\nData structure summary:")
    print(f"- Columns: {list(cleaned_df_pv.columns)}")
    print(f"- Time series type: Monthly (12 data points)")
    print(f"- Data types: {dict(cleaned_df_pv.dtypes)}")
    print(f"- Seasonal range: {summer_avg/winter_avg:.2f}x variation from winter to summer")
    
    # insights
    print(f"\nSolar Energy Data Summary:")
    print(f"- Highest solar months: {', '.join(cleaned_df_pv.nlargest(3, 'maxPv')['monthName'].tolist())}")
    print(f"- Lowest solar months: {', '.join(cleaned_df_pv.nsmallest(3, 'maxPv')['monthName'].tolist())}")
    
else:
    print("No cleaned data available to save")
    print("Troubleshooting steps:")
    print("   1. Ensure monthly-pv.csv exists in data/raw/")
    print("   2. Check file has correct columns: month, max, min, mean")
    print("   3. Verify data covers all 12 months")

IndexError: list index out of range

### pv_area.csv preparation (i.e., % area with different pv conditions - "Excellent (4+5)","Favorable (3.5-4.5)","Less than Favorable (<3.5)")
### Observable Notebook functions/charts:
#### 1.) "plot_pv_area" / "chart_pv_area" (i.e., Percentage of Area with different photovoltaic potential conditions, "Excellent (4.5+)", "Favorable (3.5-4.5)", and "Less than Favorable (<3.5)")


In [35]:
# PHOTOVOLTAIC POTENTIAL - pv_area.csv preparation from raw tif data for Observable Notebook plot functions/charts:
# 1.) "plot_pv_area"/"chart_pv_area" (i.e., Percentage of Area with different photovoltaic potential conditions, "Excellent (4.5+)", "Favorable (3.5-4.5)", and "Less than Favorable (<3.5)")

# load "raw" (i.e. "dirty") tif data


input_tif_path = get_file_by_topic('solar.', raster, spatial_dir)


# input_tif_path = 'data/raw/2025-04-colombia-cartagena_02-process-output_spatial_cartagena_solar.tif' # updatefile path


print("="*60)
print("photovoltaic potential by area data processing")
print("="*60)

# process tif file using clean_pv_area function
try:
    cleaned_df_pv = clean_pv_area(input_tif_path)
    print("photovoltaic potential by area data processed successfully!")
    
    # cleaned data structure
    print(f"\nCleaned data shape: {cleaned_df_pv.shape}")
    print(f"Cleaned data columns: {list(cleaned_df_pv.columns)}")
    print(f"\nProcessed PV data:")
    print(cleaned_df_pv)
    
    # basic data validation
    print(f"\nData Validation:")
    print(f"- Missing values: {cleaned_df_pv.isnull().sum().sum()}")
    print(f"- PV potential categories: {len(cleaned_df_pv)}")
    print(f"- Total pixels: {cleaned_df_pv['count'].sum():,.0f}")
    print(f"- Percentage sum: {cleaned_df_pv['percentage'].sum():.1f}% (should be ~100%)")
    
    # pv potential distribution analysis
    print(f"\nPV Potential Distribution:")
    
    # basic statistics
    total_pixels = cleaned_df_pv['count'].sum()
    
    for idx, row in cleaned_df_pv.iterrows():
        print(f"- {row['condition']} ({row['bin']}): {row['count']:,.0f} pixels ({row['percentage']:.1f}%)")
    
    # ID most and least favorable areas
    if len(cleaned_df_pv) > 0:
        max_coverage = cleaned_df_pv.loc[cleaned_df_pv['percentage'].idxmax()]
        min_coverage = cleaned_df_pv.loc[cleaned_df_pv['percentage'].idxmin()]
        
        print(f"\n- Dominant condition: {max_coverage['condition']} ({max_coverage['percentage']:.1f}%)")
        print(f"- Least common condition: {min_coverage['condition']} ({min_coverage['percentage']:.1f}%)")
    
    # coverage thresholds
    excellent_coverage = cleaned_df_pv[cleaned_df_pv['condition'] == 'Excellent']['percentage'].sum()
    favorable_coverage = cleaned_df_pv[cleaned_df_pv['condition'] == 'Favorable']['percentage'].sum()
    unfavorable_coverage = cleaned_df_pv[cleaned_df_pv['condition'] == 'Less than Favorable']['percentage'].sum()
    
    print(f"\nPV Potential Summary:")
    print(f"- Excellent potential areas: {excellent_coverage:.1f}%")
    print(f"- Favorable potential areas: {favorable_coverage:.1f}%")
    print(f"- Less favorable areas: {unfavorable_coverage:.1f}%")
    
    # data quality checks
    print(f"\nData Quality Checks:")
    
    quality_issues = 0
    
    # check for missing values
    missing_values = cleaned_df_pv.isnull().sum().sum()
    if missing_values > 0:
        print(f"Missing values detected: {missing_values}")
        quality_issues += 1
    
    # check for negative values (note: should NOT exist for pv potential)
    negative_counts = (cleaned_df_pv['count'] < 0).sum()
    negative_percentages = (cleaned_df_pv['percentage'] < 0).sum()
    
    if negative_counts > 0:
        print(f"Negative count values: {negative_counts}")
        quality_issues += 1
    if negative_percentages > 0:
        print(f"Negative percentage values: {negative_percentages}")
        quality_issues += 1
    
    # check percentage sum
    percentage_sum = cleaned_df_pv['percentage'].sum()
    if abs(percentage_sum - 100) > 0.1:
        print(f"Percentage sum deviation: {percentage_sum:.1f}% (should be ~100%)")
        quality_issues += 1
    
    # check for duplicate conditions
    duplicates = cleaned_df_pv['condition'].duplicated().sum()
    if duplicates > 0:
        print(f"Duplicate conditions: {duplicates}")
        quality_issues += 1
    
    if quality_issues == 0:
        print("No data quality issues detected")
    
    # data structure summary
    print(f"\nData structure summary:")
    print(f"- Columns: {list(cleaned_df_pv.columns)}")
    print(f"- Categories: {len(cleaned_df_pv)} PV potential levels")
    print(f"- Data types: {dict(cleaned_df_pv.dtypes)}")
    
    print(f"\n Output file ready: data/processed/pv_area.csv")
  
except Exception as e:
    print(f"Error processing pv potential data: {e}")
    print("Troubleshooting steps:")
    print("   1. Ensure TIF file exists at the specified path")
    print("   2. Check that the TIF file contains valid photovoltaic potential data")
    print("   3. Verify the TIF file is not corrupted")
    print("   4. Ensure rasterio library is installed: pip install rasterio")
    print("   5. Check file permissions and disk space")

print("\n" + "="*60)

photovoltaic potential by area data processing
Cleaned PV data saved to: data/processed/pv_area.csv
PV potential bins: 3
Total pixels analyzed: 717
Percentage coverage verification: 100.0% (should be ~100%)
Dominant PV condition: Favorable - 3.5-4.5 (100.0%)
photovoltaic potential by area data processed successfully!

Cleaned data shape: (3, 4)
Cleaned data columns: ['bin', 'condition', 'count', 'percentage']

Processed PV data:
       bin            condition  count  percentage
0     <3.5  Less than Favorable      0         0.0
1  3.5-4.5            Favorable    717       100.0
2     >4.5            Excellent      0         0.0

Data Validation:
- Missing values: 0
- PV potential categories: 3
- Total pixels: 717
- Percentage sum: 100.0% (should be ~100%)

PV Potential Distribution:
- Less than Favorable (<3.5): 0 pixels (0.0%)
- Favorable (3.5-4.5): 717 pixels (100.0%)
- Excellent (>4.5): 0 pixels (0.0%)

- Dominant condition: Favorable (100.0%)
- Least common condition: Less than Fa

### AIR QUALITY

### aq_area.csv preparation (i.e., % area with different air quality, 2019 concentration levels of PM2.5 (µg/mˆ3) - [0-5), [5-10), [10-15), [15-20), [20-30), [30-40), [40-50, [50-100), and [100+])
### Observable Notebook functions/charts:
#### 1.) "plot_aq_area" / "chart_aq_area" (i.e., Percentage of area with different air quality, 2019 concentration levels of PM2.5 (µg/mˆ3) - [0-5), [5-10), [10-15), [15-20), [20-30), [30-40), [40-50), [50-100), and [100+])

In [37]:
# AIR QUALITY - aq_area.csv preparation from raw tif data for Observable Notebook plot functions/charts:
# 1.) "plot_aq_area"/"chart_aq_area" (i.e., Percentage of Area with different air quality, 2019 concentration levels of PM2.5 (µg/mˆ3) - [0-5), [5-10), [10-15), [15-20), [20-30), [30-40), [40-50), [50-100), and [100+])

# load "raw" (i.e. "dirty") tif data
# input_tif_path = 'data/raw/2025-04-colombia-cartagena_02-process-output_spatial_cartagena_air.tif'

input_tif_path = get_file_by_topic('air.', raster, spatial_dir)


print("="*60)
print("AIR QUALITY (PM2.5) DATA PROCESSING")
print("="*60)

print("Analyzing TIF data structure...")

try:
    import rasterio
    with rasterio.open(input_tif_path) as src:
        data = src.read(1)
        # remove "NaN" and "NoData" values for analysis
        if src.nodata is not None:
            valid_data = data[data != src.nodata]
        else:
            valid_data = data[~np.isnan(data)]
        
        # remove negative values (i.e., shouldn't exist for PM2.5)
        valid_data = valid_data[valid_data >= 0]
        
        print(f"PM2.5 concentration range: {valid_data.min():.2f} - {valid_data.max():.2f}")
        print(f"NoData value: {src.nodata}")
        print(f"Total valid pixels: {len(valid_data):,}")
        print(f"Unique concentration values: {len(np.unique(valid_data)):,}")
        
        # basic statistics
        print(f"Mean concentration: {valid_data.mean():.2f} μg/m³")
        print(f"Median concentration: {np.median(valid_data):.2f} μg/m³")
        print(f"Standard deviation: {valid_data.std():.2f} μg/m³")

except Exception as e:
    print(f"Could not examine TIF structure: {e}")

print("\n" + "-"*40)

# process the tif file using clean_aq_area function in clean.py
try:
    cleaned_df_aq = clean_aq_area(input_tif_path)
    print("Air quality data processed successfully!")
    
    # cleaned data structure
    print(f"\nCleaned data shape: {cleaned_df_aq.shape}")
    print(f"Cleaned data columns: {list(cleaned_df_aq.columns)}")
    print(f"\nProcessed Air Quality data:")
    print(cleaned_df_aq)
    
    # basic data validation
    print(f"\nData Validation:")
    print(f"- Missing values: {cleaned_df_aq.isnull().sum().sum()}")
    print(f"- Concentration bins: {len(cleaned_df_aq)}")
    print(f"- Total pixels: {cleaned_df_aq['count'].sum():,.0f}")
    print(f"- Percentage sum: {cleaned_df_aq['percentage'].sum():.1f}% (should be ~100%)")
    
    # PM2.5 concentration distribution analysis
    print(f"\nPM2.5 Concentration Distribution:")
    
    total_pixels = cleaned_df_aq['count'].sum()
    
    for idx, row in cleaned_df_aq.iterrows():
        if row['count'] > 0:  # only show bins with data
            print(f"- {row['bin']} μg/m³: {row['count']:,.0f} pixels ({row['percentage']:.1f}%)")
    
    # air quality level analysis
    if len(cleaned_df_aq) > 0:
        # filter out zero-count categories
        active_bins = cleaned_df_aq[cleaned_df_aq['count'] > 0]
        
        if len(active_bins) > 0:
            max_concentration_bin = active_bins.loc[active_bins['percentage'].idxmax()]
            min_concentration_bin = active_bins.loc[active_bins['percentage'].idxmin()]
            
            print(f"\n- Most common concentration range: {max_concentration_bin['bin']} μg/m³ ({max_concentration_bin['percentage']:.1f}%)")
            print(f"- Least common concentration range: {min_concentration_bin['bin']} μg/m³ ({min_concentration_bin['percentage']:.1f}%)")
    
    # air quality level groupings
    good_air = cleaned_df_aq[cleaned_df_aq['bin'].isin(['[0-5)', '[5-10)'])]['percentage'].sum()
    moderate_air = cleaned_df_aq[cleaned_df_aq['bin'].isin(['[10-15)', '[15-20)'])]['percentage'].sum()
    unhealthy_air = cleaned_df_aq[cleaned_df_aq['bin'].isin(['[20-30)', '[30-40)', '[40-50)', '[50-100)', '100+'])]['percentage'].sum()
    
    print(f"\nAir Quality Summary:")
    print(f"- Lower concentrations (0-10 μg/m³): {good_air:.1f}%")
    print(f"- Moderate concentrations (10-20 μg/m³): {moderate_air:.1f}%")
    print(f"- Higher concentrations (20+ μg/m³): {unhealthy_air:.1f}%")
    
    # data quality checks
    print(f"\nData Quality Checks:")
    
    quality_issues = 0
    
    # check for missing values
    missing_values = cleaned_df_aq.isnull().sum().sum()
    if missing_values > 0:
        print(f"Missing values detected: {missing_values}")
        quality_issues += 1
    
    # check for negative values (i.e., should not exist)
    negative_counts = (cleaned_df_aq['count'] < 0).sum()
    negative_percentages = (cleaned_df_aq['percentage'] < 0).sum()
    
    if negative_counts > 0:
        print(f"Negative count values: {negative_counts}")
        quality_issues += 1
    if negative_percentages > 0:
        print(f"Negative percentage values: {negative_percentages}")
        quality_issues += 1
    
    # check percentage sum
    percentage_sum = cleaned_df_aq['percentage'].sum()
    if abs(percentage_sum - 100) > 0.1:
        print(f"Percentage sum deviation: {percentage_sum:.1f}% (should be ~100%)")
        quality_issues += 1
    
    # check for duplicate bins
    duplicates = cleaned_df_aq['bin'].duplicated().sum()
    if duplicates > 0:
        print(f"Duplicate concentration bins: {duplicates}")
        quality_issues += 1
    
    # check for expected number of bins (should be 9)
    if len(cleaned_df_aq) != 9:
        print(f"Unexpected number of bins: {len(cleaned_df_aq)} (expected 9)")
        quality_issues += 1
    
    if quality_issues == 0:
        print("No data quality issues detected")
    
except Exception as e:
    print(f"Error processing air quality data: {e}")
    print("Troubleshooting steps:")
    print("   1. Ensure TIF file exists at the specified path")
    print("   2. Check that the TIF file contains PM2.5 concentration values")
    print("   3. Verify the TIF file is not corrupted")
    print("   4. Ensure rasterio library is installed: pip install rasterio")
    print("   5. Check file permissions and disk space")
    print("   6. Verify concentration values are realistic (0-500+ μg/m³)")

print("\n" + "="*60)

AIR QUALITY (PM2.5) DATA PROCESSING
Analyzing TIF data structure...
PM2.5 concentration range: 12.70 - 17.90
NoData value: -3.4028234663852886e+38
Total valid pixels: 464
Unique concentration values: 53
Mean concentration: 15.16 μg/m³
Median concentration: 15.20 μg/m³
Standard deviation: 1.33 μg/m³

----------------------------------------
PM2.5 data range: 12.70 - 17.90 μg/m³
Unique values in data: 53
Cleaned air quality data saved to: data/processed/aq_area.csv
PM2.5 concentration bins: 9
Total pixels analyzed: 464
Percentage coverage verification: 100.0% (should be ~100%)
Most common PM2.5 range: 15-20 μg/m³ (53.2%)
Air quality data processed successfully!

Cleaned data shape: (9, 3)
Cleaned data columns: ['bin', 'count', 'percentage']

Processed Air Quality data:
      bin  count  percentage
0     0-5      0        0.00
1    5-10      0        0.00
2   10-15    217       46.77
3   15-20    247       53.23
4   20-30      0        0.00
5   30-40      0        0.00
6   40-50      0   

### Green Space, NDVI (Normalized Difference Vegetated Index)

### ndvi_area.csv preparation (i.e., percentage area with different NDVI - - i.e., "Water", [-1-0.015); "Built-up", [0.015-0.14); "Barren", [0.14-0.18); "Shrub and Grassland", [0.18-0.27); "Sparse", [0.27-0.36); and "Dense",   [0.36-1])
### Observable Notebook functions/charts:
#### 1.) "plot_ndvi_area" / "chart_ndvi_area" (i.e., percentage area with different NDVI - - i.e., "Water", [-1-0.015); "Built-up", [0.015-0.14); "Barren", [0.14-0.18); "Shrub and Grassland", [0.18-0.27); "Sparse", [0.27-0.36); and "Dense",   [0.36-1])

In [38]:
# GREEN SPACE, NDVI (Normalized Difference Vegetated Index) - ndvi_area.csv preparation from raw tif data for Observable Notebook plot functions/charts:
# 1.) "plot_ndvi_area"/"chart_ndvi_area" (i.e., percentage area with different NDVI - - i.e., "Water", [-1-0.015); "Built-up", [0.015-0.14); "Barren", [0.14-0.18); "Shrub and Grassland", [0.18-0.27); "Sparse", [0.27-0.36); and "Dense",   [0.36-1])

# load "raw" (i.e. "dirty") tif data
# input_tif_path = 'data/raw/2025-04-colombia-cartagena_02-process-output_spatial_cartagena_ndvi_season.tif'

input_tif_path = get_file_by_topic('ndvi_season', raster, spatial_dir)

print("="*60)
print("NDVI GREEN SPACE DATA PROCESSING")
print("="*60)

print("Analyzing TIF data structure...")

try:
    import rasterio
    with rasterio.open(input_tif_path) as src:
        data = src.read(1)
        # remove "NaN" and "NoData" values for analysis
        if src.nodata is not None:
            valid_data = data[data != src.nodata]
        else:
            valid_data = data[~np.isnan(data)]
        
        # remove infinite values
        valid_data = valid_data[np.isfinite(valid_data)]
        
        print(f"NDVI range: {valid_data.min():.3f} - {valid_data.max():.3f}")
        print(f"NoData value: {src.nodata}")
        print(f"Total valid pixels: {len(valid_data):,}")
        print(f"Unique NDVI values: {len(np.unique(valid_data)):,}")
        
        # show some basic statistics
        print(f"Mean NDVI: {valid_data.mean():.3f}")
        print(f"Median NDVI: {np.median(valid_data):.3f}")
        print(f"Standard deviation: {valid_data.std():.3f}")

except Exception as e:
    print(f"Could not examine TIF structure: {e}")

print("\n" + "-"*40)

# process the tif file using "clean_ndvi_area" function in clean.py
try:
    cleaned_df_ndvi = clean_ndvi_area(input_tif_path)
    print("NDVI data processed successfully!")
    
    # cleaned data structure
    print(f"\nCleaned data shape: {cleaned_df_ndvi.shape}")
    print(f"Cleaned data columns: {list(cleaned_df_ndvi.columns)}")
    print(f"\nProcessed NDVI data:")
    print(cleaned_df_ndvi)
    
    # basic data validation
    print(f"\nData Validation:")
    print(f"- Missing values: {cleaned_df_ndvi.isnull().sum().sum()}")
    print(f"- Vegetation categories: {len(cleaned_df_ndvi)}")
    print(f"- Total pixels: {cleaned_df_ndvi['count'].sum():,.0f}")
    print(f"- Percentage sum: {cleaned_df_ndvi['percentage'].sum():.1f}% (should be ~100%)")
    
    # NDVI vegetation distribution analysis
    print(f"\nVegetation Cover Distribution:")
    
    total_pixels = cleaned_df_ndvi['count'].sum()
    
    for idx, row in cleaned_df_ndvi.iterrows():
        if row['count'] > 0:  # only show categories with data
            print(f"- {row['type']} {row['bin']}: {row['count']:,.0f} pixels ({row['percentage']:.1f}%)")
    
    # vegetation type analysis
    if len(cleaned_df_ndvi) > 0:
        # filter out zero-count categories
        active_categories = cleaned_df_ndvi[cleaned_df_ndvi['count'] > 0]
        
        if len(active_categories) > 0:
            max_vegetation = active_categories.loc[active_categories['percentage'].idxmax()]
            min_vegetation = active_categories.loc[active_categories['percentage'].idxmin()]
            
            print(f"\n- Most common cover: {max_vegetation['type']} ({max_vegetation['percentage']:.1f}%)")
            print(f"- Least common cover: {min_vegetation['type']} ({min_vegetation['percentage']:.1f}%)")
    
    # green space analysis
    dense_vegetation = cleaned_df_ndvi[cleaned_df_ndvi['type'] == 'Dense']['percentage'].sum()
    sparse_vegetation = cleaned_df_ndvi[cleaned_df_ndvi['type'] == 'Sparse']['percentage'].sum()
    shrub_grassland_vegetation = cleaned_df_ndvi[cleaned_df_ndvi['type'] == 'Shrub and Grassland']['percentage'].sum()
    total_green_space = dense_vegetation + sparse_vegetation + shrub_grassland_vegetation

    # non green space analysis
    barren_space = cleaned_df_ndvi[cleaned_df_ndvi['type'] == 'Barren']['percentage'].sum()
    built_up_space = cleaned_df_ndvi[cleaned_df_ndvi['type'] == 'Built-up']['percentage'].sum()
    water_space = cleaned_df_ndvi[cleaned_df_ndvi['type'] == 'Water']['percentage'].sum()
    total_non_green_space = barren_space + built_up_space + water_space

    
    print(f"\nGreen Space Analysis:")
    print(f"- Dense vegetation coverage: {dense_vegetation:.1f}%")
    print(f"- Sparse vegetation coverage: {sparse_vegetation:.1f}%")
    print(f"- Shrub and Grassland vegetation coverage: {shrub_grassland_vegetation:.1f}%")
    print(f"- Total green space coverage: {total_green_space:.1f}%")

    print(f"\nNon Green Space Analysis:")
    print(f"- Barren coverage: {barren_space:.1f}%")
    print(f"- Built-up coverage: {built_up_space:.1f}%")
    print(f"- Water coverage: {water_space:.1f}%")
    print(f"- Total non green space coverage: {total_non_green_space:.1f}%")
    
    # data quality checks
    print(f"\nData Quality Checks:")
    
    quality_issues = 0
    
    # check for missing values
    missing_values = cleaned_df_ndvi.isnull().sum().sum()
    if missing_values > 0:
        print(f"Missing values detected: {missing_values}")
        quality_issues += 1
    
    # check for negative values in "count"/"percentage"
    negative_counts = (cleaned_df_ndvi['count'] < 0).sum()
    negative_percentages = (cleaned_df_ndvi['percentage'] < 0).sum()
    
    if negative_counts > 0:
        print(f"Negative count values: {negative_counts}")
        quality_issues += 1
    if negative_percentages > 0:
        print(f"Negative percentage values: {negative_percentages}")
        quality_issues += 1
    
    # check percentage sum
    percentage_sum = cleaned_df_ndvi['percentage'].sum()
    if abs(percentage_sum - 100) > 0.1:
        print(f"Percentage sum deviation: {percentage_sum:.1f}% (should be ~100%)")
        quality_issues += 1
    
    # check for duplicate categories
    duplicates_bin = cleaned_df_ndvi['bin'].duplicated().sum()
    duplicates_type = cleaned_df_ndvi['type'].duplicated().sum()
    
    if duplicates_bin > 0:
        print(f"Duplicate NDVI bins: {duplicates_bin}")
        quality_issues += 1
    if duplicates_type > 0:
        print(f"Duplicate vegetation types: {duplicates_type}")
        quality_issues += 1
    
    # check for expected number of categories (i.e., "type"s) (should be 6)
    if len(cleaned_df_ndvi) != 6:
        print(f"Unexpected number of categories: {len(cleaned_df_ndvi)} (expected 6)")
        quality_issues += 1
    
    # check for expected vegetation types
    expected_types = ['Water', 'Built-up', 'Barren', 'Shrub and Grassland', 'Sparse', 'Dense']
    actual_types = cleaned_df_ndvi['type'].tolist()
    missing_types = set(expected_types) - set(actual_types)
    if missing_types:
        print(f"Categories not present in data: {missing_types}")
    
    if quality_issues == 0:
        print("No data quality issues detected")
    
    
except Exception as e:
    print(f"Error processing NDVI data: {e}")
    print("Troubleshooting steps:")
    print("   1. Ensure TIF file exists at the specified path")
    print("   2. Check that the TIF file contains NDVI values (-1 to 1 range)")
    print("   3. Verify the TIF file is not corrupted")
    print("   4. Ensure rasterio library is installed: pip install rasterio")
    print("   5. Check file permissions and disk space")
    print("   6. Verify NDVI values are realistic (-1.0 to 1.0 range)")

print("\n" + "="*60)

NDVI GREEN SPACE DATA PROCESSING
Analyzing TIF data structure...
NDVI range: -0.356 - 0.680
NoData value: -9999.0
Total valid pixels: 5,233,235
Unique NDVI values: 3,911,588
Mean NDVI: 0.086
Median NDVI: 0.084
Standard deviation: 0.115

----------------------------------------
NDVI data range: -0.356 - 0.680
Unique values in data: 3911588
Cleaned NDVI data saved to: data/processed/ndvi_area.csv
NDVI vegetation categories: 6
Total pixels analyzed: 5,233,235
Percentage coverage verification: 100.0% (should be ~100%)
NDVI data processed successfully!

Cleaned data shape: (6, 4)
Cleaned data columns: ['bin', 'type', 'count', 'percentage']

Processed NDVI data:
          bin                 type    count  percentage
0    -1-0.015                Water   599907       11.46
1  0.015-0.14             Built-up  3351237       64.04
2   0.14-0.18               Barren   519999        9.94
3   0.18-0.27  Shrub and Grassland   490450        9.37
4   0.27-0.36               Sparse   184157        3.52

### Forests and Deforestation

### deforestation_area.csv preparation (i.e., percentage area that is forested and deforested (by year)
### Observable Notebook functions/charts:
#### 1.) "plot_x_area" / "chart_x_area""
....

In [39]:
# FORESTS AND DEFORESTATION - deforestation_area.csv preparation for Observable Notebook plot functions/charts:
# 1.) "plot_x_area"/"chart_x_area" (i.e., ............)

# # load "raw" (i.e. "dirty") tif data
# forest_tif_path = 'data/raw/2025-04-colombia-cartagena_02-process-output_spatial_cartagena_forest_cover23.tif'
# deforestation_tif_path = 'data/raw/2025-04-colombia-cartagena_02-process-output_spatial_cartagena_deforestation.tif'

forest_tif_path = get_file_by_topic('forest_cover23.', raster, spatial_dir)
deforestation_tif_path = get_file_by_topic('deforestation.', raster, spatial_dir)

print("="*60)
print("FOREST COVER & DEFORESTATION DATA PROCESSING")
print("="*60)

# process tif files using "clean_deforestation_area" function in clean.py
# "base_year=2000" means that value "1" = 2001; and  value "23" = 2023
# note: "auto_align=True" will automatically fix any alignment issues
try:
    cleaned_df_deforest = clean_deforestation_area(
        forest_tif_file=forest_tif_path,
        deforestation_tif_file=deforestation_tif_path,
        base_year=2000,
        auto_align=True
    )
    print("\nDeforestation data processed successfully!")
    
    # cleaned data structure
    print(f"\nYear-over-Year Deforestation Data:")
    print(cleaned_df_deforest.to_string(index=False))
    
    # data validation
    print(f"\n" + "="*60)
    print("DATA VALIDATION:")
    print(f"- Total years tracked: {len(cleaned_df_deforest)}")
    print(f"- Missing values: {cleaned_df_deforest.isnull().sum().sum()}")
    
    # check that percentages add up correctly (should be 100%)
    final_row = cleaned_df_deforest.iloc[-1]
    percent_check = final_row['percent_forest_remaining'] + final_row['percent_forest_lost']
    print(f"- Percent remaining + lost: {percent_check:.1f}% (should be 100%)")
    
    # deforestation trend analysis
    print(f"\n" + "="*60)
    print("DEFORESTATION TRENDS:")
    
    years_with_deforest = cleaned_df_deforest[cleaned_df_deforest['deforested_this_year'] > 0]
    
    if len(years_with_deforest) > 0:
        # overall statistics
        total_deforested = final_row['cumulative_deforested']
        avg_annual = years_with_deforest['deforested_this_year'].mean()
        
        print(f"- Years with deforestation: {len(years_with_deforest)}")
        print(f"- Average annual loss: {avg_annual:.0f} pixels/year")
        print(f"- Total forest lost: {total_deforested:,} pixels ({final_row['percent_forest_lost']:.2f}%)")
        
        # peak year of deforestation
        peak = years_with_deforest.loc[years_with_deforest['deforested_this_year'].idxmax()]
        print(f"- Peak deforestation: {peak['year']} ({peak['deforested_this_year']:,} pixels)")
        
        # recent deforestation trend (i.e., last 5 years)
        recent_5 = years_with_deforest.tail(5)
        recent_total = recent_5['deforested_this_year'].sum()
        recent_avg = recent_5['deforested_this_year'].mean()
        
        print(f"\nRecent 5 Years (last available data):")
        print(f"- Total deforested: {recent_total:,} pixels")
        print(f"- Average annual: {recent_avg:.0f} pixels/year")
        
        # compare recent deforestation vs overall average deforestation
        # note:
        # ratio = recent 5 years (pixel/year) / overall historical average (pixels/year), example: 13(pixel/year) / 18(pixel/year) = 0.7x overall average - SLOWING!

        # ratio is > 1, recent deforestaion is FASTER than historic average deforestation (i.e., accelerating);
        # ratio is < 1, recent deforestaion is SLOWER than historic average deforestation (i.e., decelerating);
        # ratio is = 1, recent deforestaion is at the SAME rate as historic average deforestation (i.e., steady);
        if recent_avg > avg_annual:
            print(f"- Trend: Accelerating ({recent_avg/avg_annual:.1f}x overall average)")
        else:
            print(f"- Trend: Slowing ({recent_avg/avg_annual:.1f}x overall average)")
    
    # data quality checks
    print(f"\n" + "="*60)
    print("DATA QUALITY CHECKS:")
    
    quality_issues = 0
    
    # check for negative values
    if (cleaned_df_deforest['forest_remaining'] < 0).any():
        print("Negative forest remaining values detected")
        quality_issues += 1
    
    # check that cumulative never decreases
    cumulative_diff = cleaned_df_deforest['cumulative_deforested'].diff()
    if (cumulative_diff[cumulative_diff.notna()] < 0).any():
        print("Cumulative deforestation decreases (impossible)")
        quality_issues += 1
    
    # check that forest remaining + cumulative = baseline
    baseline = cleaned_df_deforest.iloc[0]['forest_remaining']
    for idx, row in cleaned_df_deforest.iterrows():
        if row['forest_remaining'] + row['cumulative_deforested'] != baseline:
            print(f"Year {row['year']}: Forest accounting error")
            quality_issues += 1
            break
    
    if quality_issues == 0:
        print("No data quality issues detected")
    
except Exception as e:
    print(f"Error processing deforestation data: {e}")
    print("\nTroubleshooting:")
    print("1. Check file paths are correct")
    print("2. Ensure rasterio is installed: pip install rasterio")
    print("3. Verify TIF files are valid and not corrupted")

print("\n" + "="*60)

FOREST COVER & DEFORESTATION DATA PROCESSING
Alignment check:
- Same CRS: True
- Same shape: True
- Same bounds: True
- Fully aligned: True
TIFs are properly aligned

Baseline forest area: 5,339 pixels

Cleaned deforestation data saved to: data/processed/deforestation_area.csv
Time period: 2000 - 2012.0
Baseline forest: 5,339 pixels
Total deforested: 27.0 pixels (0.51%)
Forest remaining: 5,312.0 pixels (99.49%)
Peak deforestation year: 2008.0 (10.0 pixels)

Deforestation data processed successfully!

Year-over-Year Deforestation Data:
 year  forest_remaining  deforested_this_year  cumulative_deforested  percent_forest_remaining  percent_forest_lost
 2000              5339                     0                      0                    100.00                 0.00
 2003              5335                     4                      4                     99.93                 0.07
 2005              5333                     2                      6                     99.89                 

# RISK IDENTIFICATION

### FLOOD EVENTS

### fu.csv, pu.csv, cu.csv, and comb.csv preparation
### Observable Notebook functions/charts:
#### 1.) "plot_fu" / "chart_fu" (i.e., built-up area exposed to river (fluvial) flooding)
#### 2.) "plot_pu" / "chart_pu" (i.e., built-up area exposed to rainwater (pluvial) flooding)
#### 3.) "plot_cu" / "chart_cu" (i.e., built-up area exposed to coastal flooding)
#### 4.) "plot_comb" / "chart_comb" (i.e., built-up area exposed to combined flooding)

#### NOTE:
#### 5.) data, raw, "flood-events.csv" is already ready for Observable Notebook "plot_fe" / "chart_fe" (i.e.,large flood events in city, country)

In [46]:
# FLOODING - Multiple csv preparation for Observable Notebook plot functions/charts:
# 1.) "plot_fu"/"chart_fu" (i.e., built-up area exposed to river (fluvial) flooding)
# 2.) "plot_pu"/"chart_pu" (i.e., built-up area exposed to rainwater (pluvial) flooding)
# 3.) "plot_cu"/"chart_cu" (i.e., built-up area exposed to coastal flooding)
# 4.) "plot_comb"/"chart_comb" (i.e., built-up area exposed to combined flooding)
#5.) NOTE: data, raw, "flood-events.csv" is already ready for Observable Notebook "plot_fe" / "chart_fe" (i.e.,large flood events in city, country)


# load "raw" (i.e. "dirty") tabular output data
# NOTE: Flood data structure may vary by city - (i.e., coastal cities have coastal flood data, inland cities do not)

filename = get_file_by_topic('flood_wsf', tabular, tabular_dir)

raw_df_flood = pd.read_csv(filename) # updatefile path

# basic info about raw data
print("Raw flood risk data info:")
print(f"Shape: {raw_df_flood.shape}")
print(f"Columns: {list(raw_df_flood.columns)}")
print(f"Year range: {raw_df_flood['year'].min()} - {raw_df_flood['year'].max()}")
print(f"Total data points: {len(raw_df_flood)}")

# ID available flood types
flood_columns = [col for col in raw_df_flood.columns if '_2020' in col]
print(f"Available flood types: {flood_columns}")

# preview flood risk ranges for each type
for col in flood_columns:
    flood_type = col.replace('_2020', '')
    print(f"{flood_type.capitalize()} risk range: {raw_df_flood[col].min():.2f} - {raw_df_flood[col].max():.2f}")

print(f"Data preview:")
print(raw_df_flood.head())
print("\n" + "="*50 + "\n")

# clean data using clean_flood function in clean.py
try:
    created_files = clean_flood(filename) # updatefile path
    print("Flood risk data processed successfully!")
    
    # load and validate each created file
    flood_dataframes = {}
    
    for filename in created_files:
        file_path = f'data/processed/{filename}'
        flood_type = filename.replace('.csv', '')
        
        try:
            df = pd.read_csv(file_path)
            flood_dataframes[flood_type] = df
            
            print(f"\n{filename} validation:")
            print(f"- Shape: {df.shape}")
            print(f"- Columns: {list(df.columns)}")
            print(f"- Year range: {df['yearName'].min()} - {df['yearName'].max()}")
            print(f"- Risk range: {df.iloc[:, 2].min():.2f} - {df.iloc[:, 2].max():.2f}")  # Note: third column is risk value
            print(f"- Sample data:")
            print(df.head(5))
            
        except Exception as e:
            print(f"Error loading {filename}: {e}")
    
    # basic flood risk analysis
    if len(flood_dataframes) > 0:
        print(f"\nFlood Risk Data Summary:")
        
        for flood_type, df in flood_dataframes.items():
            risk_column = df.columns[2]  # Note: third column contains risk values
            
            # basic statistics
            avg_risk = df[risk_column].mean()
            max_risk = df[risk_column].max()
            min_risk = df[risk_column].min()
            std_risk = df[risk_column].std()
            
            # trend calculation
            trend = df[risk_column].iloc[-1] - df[risk_column].iloc[0]
            
            print(f"\n- {flood_type.upper()} flood risk:")
            print(f"  Average: {avg_risk:.2f}")
            print(f"  Range: {min_risk:.2f} - {max_risk:.2f}")
            print(f"  Standard deviation: {std_risk:.2f}")
            print(f"  Change (1985-2015): {trend:+.2f}")
        
        # current values
        if len(flood_dataframes) > 1:
            print(f"\n2015 Risk Values:")
            for flood_type, df in flood_dataframes.items():
                risk_column = df.columns[2]
                current_risk = df[risk_column].iloc[-1]
                print(f"- {flood_type.upper()}: {current_risk:.2f}")
        
        # data quality checks
        print(f"\nData Quality Checks:")
        
        quality_issues = 0
        for flood_type, df in flood_dataframes.items():
            risk_column = df.columns[2]
            
            # check for missing values
            missing_values = df[risk_column].isna().sum()
            if missing_values > 0:
                print(f"{flood_type.upper()}: {missing_values} missing values")
                quality_issues += 1
            
            # check for negative values (mathematically impossible for risk)
            negative_values = (df[risk_column] < 0).sum()
            if negative_values > 0:
                print(f"{flood_type.upper()}: {negative_values} negative risk values")
                quality_issues += 1
        
        if quality_issues == 0:
            print("No data quality issues detected")
    
except Exception as e:
    print(f"Error processing flood risk data: {e}")
    print("Check that the flood CSV file exists and has the correct format")
    print("Expected columns: year, coastal_2020, fluvial_2020, pluvial_2020, comb_2020 (as available)")

# save confirmation and next steps
if 'created_files' in locals() and len(created_files) > 0:
    print(f"\n Cleaned data saved to data/processed/:")
    for filename in created_files:
        print(f"    {filename}")
    
    # data structure summary
    print(f"\n Data structure summary:")
    print(f"- Files created: {len(created_files)}")
    print(f"- Time series length: {len(list(flood_dataframes.values())[0]) if flood_dataframes else 'N/A'} years")
    print(f"- Year range: {raw_df_flood['year'].min()} - {raw_df_flood['year'].max()}")
    print(f"- Data types: {dict(list(flood_dataframes.values())[0].dtypes) if flood_dataframes else 'N/A'}")
    
else:
    print("No cleaned flood data available")
    print("Troubleshooting steps:")
    print("   1. Ensure flood CSV file exists in data/raw/")
    print("   2. Check file has correct columns with '_2020' suffix")
    print("   3. Verify data covers expected time range (1985-2015)")
    print("   4. Check that at least one flood type column exists")

Raw flood risk data info:
Shape: (31, 5)
Columns: ['year', 'coastal_2020', 'comb_2020', 'fluvial_2020', 'pluvial_2020']
Year range: 1985 - 2015
Total data points: 31
Available flood types: ['coastal_2020', 'comb_2020', 'fluvial_2020', 'pluvial_2020']
Coastal risk range: 2.27 - 5.31
Comb risk range: 32.76 - 63.92
Fluvial risk range: 3.35 - 7.67
Pluvial risk range: 29.78 - 56.92
Data preview:
   year  coastal_2020  comb_2020  fluvial_2020  pluvial_2020
0  1985      2.273944  32.761063      3.351538     29.782652
1  1986      2.545175  34.641355      3.474692     31.410039
2  1987      2.833266  36.746696      3.682880     33.228754
3  1988      2.955687  38.728150      3.977569     34.975630
4  1989      3.063446  39.978013      4.116850     36.076682


Available flood types: ['coastal', 'fluvial', 'pluvial', 'combined']
Created cu.csv: 31 records
   Year range: 1985 - 2015
   CU range: 2.27 - 5.31
Created fu.csv: 31 records
   Year range: 1985 - 2015
   FU range: 3.35 - 7.67
Created pu.

### ELEVATION

### e.csv preparation
### Observable Notebook functions/charts:
#### 1.) "plot_e" / "plot_e_alt" / "chart_e" (i.e., elevation)

In [48]:
# ELEVATION ANALYSIS - e.csv preparation for Observable Notebook plot functions/charts:
# 1.) "plot_e"/"chart_e" (elevation distribution vertical bar chart)

# load "raw" (i.e. "dirty") tabular output data

filename = get_file_by_topic('elevation', tabular, tabular_dir)

raw_df_e = pd.read_csv(filename) # updatefile path

# basic info about raw data
print("Raw elevation data info:")
print(f"Shape: {raw_df_e.shape}")
print(f"Columns: {list(raw_df_e.columns)}")
print(f"Total data points: {len(raw_df_e)}")

# preview elevation ranges and counts if available
if 'Bin' in raw_df_e.columns and 'Count' in raw_df_e.columns:
    print(f"Elevation bins: {len(raw_df_e)}")
    print(f"Total pixels: {raw_df_e['Count'].sum():,.0f}")
    print(f"Pixel count range: {raw_df_e['Count'].min():,.0f} - {raw_df_e['Count'].max():,.0f}")
    print(f"Elevation ranges: {raw_df_e['Bin'].iloc[0]} to {raw_df_e['Bin'].iloc[-1]}")

print(f"Data preview:")
print(raw_df_e.head())
print("\n" + "="*50 + "\n")

# clean data using clean_e function
try:
    cleaned_df_e = clean_e(filename) # updatefile path
    print("Elevation data cleaned successfully!")
    
    # cleaned data info
    print(f"\nCleaned data shape: {cleaned_df_e.shape}")
    print(f"Cleaned data columns: {list(cleaned_df_e.columns)}")
    print(f"Sample of cleaned data:")
    print(cleaned_df_e.head(10))
    
    # basic data validation
    print(f"\nData validation:")
    print(f"- Missing values: {cleaned_df_e.isnull().sum().sum()}")
    print(f"- Elevation bins: {len(cleaned_df_e)}")
    print(f"- Total pixels: {cleaned_df_e['count'].sum():,.0f}")
    print(f"- Percentage sum: {cleaned_df_e['percentage'].sum():.1f}% (should be ~100%)")
    
    # elevation data analysis
    print(f"\nElevation Data Summary:")
    
    # basic statistics
    avg_percentage = cleaned_df_e['percentage'].mean()
    median_percentage = cleaned_df_e['percentage'].median()
    
    print(f"- Elevation ranges present: {len(cleaned_df_e)}")
    print(f"- Pixel count range: {cleaned_df_e['count'].min():,.0f} - {cleaned_df_e['count'].max():,.0f}")
    print(f"- Percentage range: {cleaned_df_e['percentage'].min():.2f}% - {cleaned_df_e['percentage'].max():.2f}%")
    
    # ID extremes
    dominant_bin = cleaned_df_e.loc[cleaned_df_e['percentage'].idxmax()]
    least_common_bin = cleaned_df_e.loc[cleaned_df_e['percentage'].idxmin()]
    
    print(f"- Most common elevation: {dominant_bin['bin']} ({dominant_bin['percentage']:.2f}%)")
    print(f"- Least common elevation: {least_common_bin['bin']} ({least_common_bin['percentage']:.2f}%)")
    print(f"- Lowest elevation range: {cleaned_df_e['bin'].iloc[0]}")
    print(f"- Highest elevation range: {cleaned_df_e['bin'].iloc[-1]}")
    
    # coverage distribution
    above_20_percent = (cleaned_df_e['percentage'] >= 20).sum()
    above_10_percent = (cleaned_df_e['percentage'] >= 10).sum()
    above_5_percent = (cleaned_df_e['percentage'] >= 5).sum()
    
    print(f"- Ranges with ≥20% coverage: {above_20_percent}")
    print(f"- Ranges with ≥10% coverage: {above_10_percent}")
    print(f"- Ranges with ≥5% coverage: {above_5_percent}")
    
    # data quality checks
    print(f"\nData Quality Checks:")
    
    quality_issues = 0
    
    # check for missing values in key columns
    missing_bin = cleaned_df_e['bin'].isna().sum()
    missing_count = cleaned_df_e['count'].isna().sum()
    missing_percentage = cleaned_df_e['percentage'].isna().sum()
    
    if missing_bin > 0:
        print(f"Missing elevation bin values: {missing_bin}")
        quality_issues += 1
    if missing_count > 0:
        print(f"Missing count values: {missing_count}")
        quality_issues += 1
    if missing_percentage > 0:
        print(f"Missing percentage values: {missing_percentage}")
        quality_issues += 1
    
    # check for impossible values
    negative_count = (cleaned_df_e['count'] < 0).sum()
    negative_percentage = (cleaned_df_e['percentage'] < 0).sum()
    
    if negative_count > 0:
        print(f"Negative count values: {negative_count}")
        quality_issues += 1
    if negative_percentage > 0:
        print(f"Negative percentage values: {negative_percentage}")
        quality_issues += 1
    
    # check percentage sum
    percentage_sum = cleaned_df_e['percentage'].sum()
    if abs(percentage_sum - 100) > 0.1:
        print(f"Percentage sum deviation: {percentage_sum:.1f}% (should be ~100%)")
        quality_issues += 1
    
    # check for duplicate elevation bins
    duplicates = cleaned_df_e['bin'].duplicated().sum()
    if duplicates > 0:
        print(f"Duplicate elevation bins: {duplicates}")
        quality_issues += 1
    
    if quality_issues == 0:
        print("No data quality issues detected")
        
except Exception as e:
    print(f"Error cleaning elevation data: {e}")
    print("Check that the elevation CSV file exists and has the correct format")
    print("Expected columns: Bin, Count")

# save confirmation and next steps
if 'cleaned_df_e' in locals():
    print(f"\nCleaned data saved to: data/processed/e.csv")
    
    # preview of data structure
    print(f"\n Data structure summary:")
    print(f"- Columns: {list(cleaned_df_e.columns)}")
    print(f"- Data points: {len(cleaned_df_e)} elevation ranges")
    print(f"- Data types: {dict(cleaned_df_e.dtypes)}")
    print(f"- Elevation coverage: {cleaned_df_e['percentage'].min():.2f}% - {cleaned_df_e['percentage'].max():.2f}%")
    
else:
    print("No cleaned elevation data available")
    print("Troubleshooting steps:")
    print("   1. Ensure elevation CSV file exists in data/raw/")
    print("   2. Check file has correct columns: Bin, Count")
    print("   3. Verify elevation bins are properly labeled")
    print("   4. Check that count values are numeric and positive")

Raw elevation data info:
Shape: (5, 2)
Columns: ['Bin', 'Count']
Total data points: 5
Elevation bins: 5
Total pixels: 549,697
Pixel count range: 890 - 413,599
Elevation ranges: -5-40 to 185-235
Data preview:
       Bin   Count
0    -5-40  413599
1    40-90   94379
2   90-135   32786
3  135-185    8043
4  185-235     890


Cleaned data saved to: data/processed/e.csv
Elevation bins: 5
Elevation range: 40-90 to -5-40
Total area analyzed: 549,697 pixels
Percentage coverage verification: 100.0% (should be ~100%)
Dominant elevation range: -5-40 (75.2%)
Major elevation ranges (≥10% coverage): 2 bins
Major ranges: 40-90, -5-40
Elevation data cleaned successfully!

Cleaned data shape: (5, 3)
Cleaned data columns: ['bin', 'count', 'percentage']
Sample of cleaned data:
       bin   count  percentage
0    40-90   94379       17.17
1   90-135   32786        5.96
2  135-185    8043        1.46
3  185-235     890        0.16
4    -5-40  413599       75.24

Data validation:
- Missing values: 0
- Eleva

### SLOPE

### s.csv preparation
### Observable Notebook functions/charts:
#### 1.) "plot_s" / "chart_s" (i.e., slope)

In [49]:
# SLOPE ANALYSIS - s.csv preparation for Observable Notebook plot functions/charts:
# 1.) "plot_s"/"chart_s" (slope chart)

# load "raw" (i.e. "dirty") tabular output data

filename = get_file_by_topic('slope', tabular, tabular_dir)
raw_df_s = pd.read_csv(filename) # updatefile path

# basic info about raw data
print("Raw slope data info:")
print(f"Shape: {raw_df_s.shape}")
print(f"Columns: {list(raw_df_s.columns)}")
print(f"Total data points: {len(raw_df_s)}")

# slope ranges and counts
if 'Bin' in raw_df_s.columns and 'Count' in raw_df_s.columns:
    print(f"Slope bins: {len(raw_df_s)}")
    print(f"Total pixels: {raw_df_s['Count'].sum():,.0f}")
    print(f"Pixel count range: {raw_df_s['Count'].min():,.0f} - {raw_df_s['Count'].max():,.0f}")
    print(f"Slope ranges: {raw_df_s['Bin'].iloc[0]} to {raw_df_s['Bin'].iloc[-1]} degrees")

print(f"Data preview:")
print(raw_df_s.head())
print("\n" + "="*50 + "\n")

# clean data using clean_s function
try:
    cleaned_df_s = clean_s(filename) # updatefile path
    print("Slope data cleaned successfully!")
    
    # cleaned data info
    print(f"\nCleaned data shape: {cleaned_df_s.shape}")
    print(f"Cleaned data columns: {list(cleaned_df_s.columns)}")
    print(f"Sample of cleaned data:")
    print(cleaned_df_s.head(10))
    
    # basic data validation
    print(f"\nData validation:")
    print(f"- Missing values: {cleaned_df_s.isnull().sum().sum()}")
    print(f"- Slope bins: {len(cleaned_df_s)}")
    print(f"- Total pixels: {cleaned_df_s['count'].sum():,.0f}")
    print(f"- Percentage sum: {cleaned_df_s['percentage'].sum():.1f}% (should be ~100%)")
    
    # slope data analysis
    print(f"\nSlope Data Summary:")
    
    print(f"- Slope ranges present: {len(cleaned_df_s)}")
    print(f"- Pixel count range: {cleaned_df_s['count'].min():,.0f} - {cleaned_df_s['count'].max():,.0f}")
    print(f"- Percentage range: {cleaned_df_s['percentage'].min():.2f}% - {cleaned_df_s['percentage'].max():.2f}%")
    
    # ID extremes
    dominant_bin = cleaned_df_s.loc[cleaned_df_s['percentage'].idxmax()]
    least_common_bin = cleaned_df_s.loc[cleaned_df_s['percentage'].idxmin()]
    
    print(f"- Most common slope: {dominant_bin['bin']} degrees ({dominant_bin['percentage']:.2f}%)")
    print(f"- Least common slope: {least_common_bin['bin']} degrees ({least_common_bin['percentage']:.2f}%)")
    print(f"- Gentlest slope range: {cleaned_df_s['bin'].iloc[0]} degrees")
    print(f"- Steepest slope range: {cleaned_df_s['bin'].iloc[-1]} degrees")
    
    # coverage distribution
    above_20_percent = (cleaned_df_s['percentage'] >= 20).sum()
    above_10_percent = (cleaned_df_s['percentage'] >= 10).sum()
    above_5_percent = (cleaned_df_s['percentage'] >= 5).sum()
    
    print(f"- Ranges with ≥20% coverage: {above_20_percent}")
    print(f"- Ranges with ≥10% coverage: {above_10_percent}")
    print(f"- Ranges with ≥5% coverage: {above_5_percent}")
    
    # data quality checks
    print(f"\nData Quality Checks:")
    
    quality_issues = 0
    
    # check for missing values in key columns
    missing_bin = cleaned_df_s['bin'].isna().sum()
    missing_count = cleaned_df_s['count'].isna().sum()
    missing_percentage = cleaned_df_s['percentage'].isna().sum()
    
    if missing_bin > 0:
        print(f"Missing slope bin values: {missing_bin}")
        quality_issues += 1
    if missing_count > 0:
        print(f"Missing count values: {missing_count}")
        quality_issues += 1
    if missing_percentage > 0:
        print(f"Missing percentage values: {missing_percentage}")
        quality_issues += 1
    
    # check for impossible values
    negative_count = (cleaned_df_s['count'] < 0).sum()
    negative_percentage = (cleaned_df_s['percentage'] < 0).sum()
    
    if negative_count > 0:
        print(f"Negative count values: {negative_count}")
        quality_issues += 1
    if negative_percentage > 0:
        print(f"Negative percentage values: {negative_percentage}")
        quality_issues += 1
    
    # check percentage sum
    percentage_sum = cleaned_df_s['percentage'].sum()
    if abs(percentage_sum - 100) > 0.1:
        print(f"Percentage sum deviation: {percentage_sum:.1f}% (should be ~100%)")
        quality_issues += 1
    
    # check for duplicate slope bins
    duplicates = cleaned_df_s['bin'].duplicated().sum()
    if duplicates > 0:
        print(f"Duplicate slope bins: {duplicates}")
        quality_issues += 1
    
    if quality_issues == 0:
        print("No data quality issues detected")
        
except Exception as e:
    print(f"Error cleaning slope data: {e}")
    print("Check that the slope CSV file exists and has the correct format")
    print("Expected columns: Bin, Count")

# save confirmation and next steps
if 'cleaned_df_s' in locals():
    print(f"\nCleaned data saved to: data/processed/s.csv")
    
    # quick preview of data structure
    print(f"\nData structure summary:")
    print(f"- Columns: {list(cleaned_df_s.columns)}")
    print(f"- Data points: {len(cleaned_df_s)} slope ranges")
    print(f"- Data types: {dict(cleaned_df_s.dtypes)}")
    print(f"- Slope coverage: {cleaned_df_s['percentage'].min():.2f}% - {cleaned_df_s['percentage'].max():.2f}%")
      
else:
    print("No cleaned slope data available")
    print("Troubleshooting steps:")
    print("   1. Ensure slope CSV file exists in data/raw/")
    print("   2. Check file has correct columns: Bin, Count")
    print("   3. Verify slope bins are properly labeled (e.g., 0-2, 2-5)")
    print("   4. Check that count values are numeric and positive")

Raw slope data info:
Shape: (5, 2)
Columns: ['Bin', 'Count']
Total data points: 5
Slope bins: 5
Total pixels: 549,702
Pixel count range: 1,057 - 428,343
Slope ranges: 0-2 to 20-90 degrees
Data preview:
     Bin   Count
0    0-2  428343
1    2-5   79034
2   5-10   31121
3  10-20   10147
4  20-90    1057


Cleaned data saved to: data/processed/s.csv
Slope bins: 5
Slope range: 0-2 to 20-90 degrees
Total area analyzed: 549,702 pixels
Percentage coverage verification: 100.0% (should be ~100%)
Dominant slope range: 0-2 degrees (77.9%)
Relatively flat areas (0-5 degrees): 79.8%
Significant slope ranges (≥5% coverage): 3 bins
Significant ranges: 0-2, 2-5, 5-10
Slope data cleaned successfully!

Cleaned data shape: (5, 3)
Cleaned data columns: ['bin', 'count', 'percentage']
Sample of cleaned data:
     bin   count  percentage
0    0-2  428343       77.92
1    2-5   79034       14.38
2   5-10   31121        5.66
3  10-20   10147        1.85
4  20-90    1057        0.19

Data validation:
- Missing

### LANDSLIDE SUSCEPTIBILITY

### ls_area.csv preparation (i.e., % area with different landslide susceptibility levels - "No Data" (0); "Very low" (1); "Low" (2); "Medium" (3); "High" (4); and "Very high" (5))
### Observable Notebook functions/charts:
#### 1.) "plot_ls_area" / "chart_ls_area" (i.e., Percentage of area with different landslide susceptibiltiy levels, "No Data" (0); "Very low" (1); "Low" (2); "Medium" (3); "High" (4); and "Very high" (5)

In [50]:
# LANDSLIDE SUSCEPTIBILITY - ls_area.csv preparation from raw tif data for Observable Notebook plot functions/charts:
# 1.) "plot_ls_area"/"chart_ls_area" (i.e., Percentage of Area with different landslide susceptibility levels, "No Data" (0); "Very low" (1); "Low" (2); "Medium" (3); "High" (4); and "Very high" (5)

# load "raw" (i.e. "dirty") tif data

# input_tif_path = 'data/raw/2025-04-colombia-cartagena_02-process-output_spatial_cartagena_landslide.tif'  # updatefile path

input_tif_path = get_file_by_topic('landslide', raster, spatial_dir)

print("="*60)
print("LANDSLIDE SUSCEPTIBILITY DATA PROCESSING")
print("="*60)

# data value distribution
print("Analyzing TIF data structure...")

try:
    import rasterio
    with rasterio.open(input_tif_path) as src:
        data = src.read(1)
        unique_vals = np.unique(data[~np.isnan(data)])
        print(f"Unique values in TIF: {unique_vals}")
        print(f"Data range: {data.min()} to {data.max()}")
        print(f"NoData value: {src.nodata}")
        
        # count pixels for each value
        for val in unique_vals:
            count = np.sum(data == val)
            print(f"Value {val}: {count:,} pixels")

except Exception as e:
    print(f"Could not examine TIF structure: {e}")

print("\n" + "-"*40)

# process tif file using clean_ls_area function in clean.py
# set "include_nodata=False" to exclude value, "0"  ("NoData"/background)
try:
    cleaned_df_landslide = clean_ls_area(input_tif_path, include_nodata=False)
    print("Landslide susceptibility data processed successfully!")
    
    # cleaned data structure
    print(f"\nCleaned data shape: {cleaned_df_landslide.shape}")
    print(f"Cleaned data columns: {list(cleaned_df_landslide.columns)}")
    print(f"\nProcessed Landslide Susceptibility data:")
    print(cleaned_df_landslide)
    
    # basic data validation
    print(f"\nData Validation:")
    print(f"- Missing values: {cleaned_df_landslide.isnull().sum().sum()}")
    print(f"- Susceptibility categories: {len(cleaned_df_landslide)}")
    print(f"- Total pixels: {cleaned_df_landslide['count'].sum():,.0f}")
    print(f"- Percentage sum: {cleaned_df_landslide['percentage'].sum():.1f}% (should be ~100%)")
    
    # landslide susceptibility distribution analysis
    print(f"\nLandslide Susceptibility Distribution:")
    
    total_pixels = cleaned_df_landslide['count'].sum()
    
    for idx, row in cleaned_df_landslide.iterrows():
        if row['count'] > 0:  # only show categories with data (i.e., exclude "0")
            print(f"- {row['bin']}: {row['count']:,.0f} pixels ({row['percentage']:.1f}%)")
    
    # risk level analysis
    if len(cleaned_df_landslide) > 0:
        # filter out zero-count categories
        active_categories = cleaned_df_landslide[cleaned_df_landslide['count'] > 0]
        
        if len(active_categories) > 0:
            max_susceptibility = active_categories.loc[active_categories['percentage'].idxmax()]
            min_susceptibility = active_categories.loc[active_categories['percentage'].idxmin()]
            
            print(f"\n- Most common susceptibility: {max_susceptibility['bin']} ({max_susceptibility['percentage']:.1f}%)")
            print(f"- Least common susceptibility: {min_susceptibility['bin']} ({min_susceptibility['percentage']:.1f}%)")
    
    # risk level groupings
    very_low_risk = cleaned_df_landslide[cleaned_df_landslide['bin'].isin(['Very low'])]['percentage'].sum()
    low_risk = cleaned_df_landslide[cleaned_df_landslide['bin'].isin(['Low'])]['percentage'].sum()
    medium_risk = cleaned_df_landslide[cleaned_df_landslide['bin'] == 'Medium']['percentage'].sum()
    high_risk = cleaned_df_landslide[cleaned_df_landslide['bin'].isin(['High'])]['percentage'].sum()
    very_high_risk = cleaned_df_landslide[cleaned_df_landslide['bin'].isin(['Very high'])]['percentage'].sum()
    
    print(f"\nRisk Level Summary:")
    print(f"- Very low risk areas: {very_low_risk:.1f}%")
    print(f"- Low risk areas: {low_risk:.1f}%")
    print(f"- Medium risk areas: {medium_risk:.1f}%")
    print(f"- High risk areas: {very_high_risk:.1f}%")
    print(f"- Very high risk areas: {high_risk:.1f}%")

    
    # data quality checks
    print(f"\nData Quality Checks:")
    
    quality_issues = 0
    
    # check for missing values
    missing_values = cleaned_df_landslide.isnull().sum().sum()
    if missing_values > 0:
        print(f"Missing values detected: {missing_values}")
        quality_issues += 1
    
    # check for negative values (should not exist)
    negative_counts = (cleaned_df_landslide['count'] < 0).sum()
    negative_percentages = (cleaned_df_landslide['percentage'] < 0).sum()
    
    if negative_counts > 0:
        print(f"Negative count values: {negative_counts}")
        quality_issues += 1
    if negative_percentages > 0:
        print(f"Negative percentage values: {negative_percentages}")
        quality_issues += 1
    
    # check percentage sum
    percentage_sum = cleaned_df_landslide['percentage'].sum()
    if abs(percentage_sum - 100) > 0.1:
        print(f"Percentage sum deviation: {percentage_sum:.1f}% (should be ~100%)")
        quality_issues += 1
    
    # check for duplicate categories
    duplicates = cleaned_df_landslide['bin'].duplicated().sum()
    if duplicates > 0:
        print(f"Duplicate susceptibility categories: {duplicates}")
        quality_issues += 1
    
    # check for expected landslide susceptibility categories
    expected_categories = ['Very low', 'Low', 'Medium', 'High', 'Very high']
    actual_categories = cleaned_df_landslide['bin'].tolist()
    missing_categories = set(expected_categories) - set(actual_categories)
    if missing_categories:
        print(f"Categories not present in data: {missing_categories}")
    
    if quality_issues == 0:
        print("No data quality issues detected")
    
except Exception as e:
    print(f"Error processing landslide susceptibility data: {e}")
    print("Troubleshooting steps:")
    print("   1. Ensure TIF file exists at the specified path")
    print("   2. Check that the TIF file contains values 1-5 (or 0-5)")
    print("   3. Verify the TIF file is not corrupted")
    print("   4. Ensure rasterio library is installed: pip install rasterio")
    print("   5. Check file permissions and disk space")
    print("   6. If data includes value 0, try setting include_nodata=True")

print("\n" + "="*60)

# optional: to include value, "0" in analysis
# print("\n" + "="*30 + " INCLUDING VALUE 0 " + "="*30)
# try:
#     cleaned_df_with_zero = clean_ls_area(input_tif_path, 
#                                                output_file='data/processed/ls_area_with_nodata.csv',
#                                                include_nodata=True)
#     print("Alternative analysis (including value, "0") completed!")
#     print(cleaned_df_with_zero)
# except Exception as e:
#     print(f"Error in alternative analysis: {e}")

LANDSLIDE SUSCEPTIBILITY DATA PROCESSING
Analyzing TIF data structure...
Unique values in TIF: [  0   1   2   3   4   5 127]
Data range: 0 to 127
NoData value: 127.0
Value 0: 37 pixels
Value 1: 414 pixels
Value 2: 215 pixels
Value 3: 38 pixels
Value 4: 12 pixels
Value 5: 1 pixels
Value 127: 765 pixels

----------------------------------------
Unique values found in data: [1 2 3 4 5]
Cleaned landslide data saved to: data/processed/ls_area.csv
Susceptibility categories: 5
Total pixels analyzed: 680
Percentage coverage verification: 100.0% (should be ~100%)
Dominant susceptibility level: Very low (60.9%)
Landslide susceptibility data processed successfully!

Cleaned data shape: (5, 4)
Cleaned data columns: ['bin', 'susceptibility', 'count', 'percentage']

Processed Landslide Susceptibility data:
         bin susceptibility  count  percentage
0   Very low              1    414       60.88
1        Low              2    215       31.62
2     Medium              3     38        5.59
3       

### EARTHQUAKE EVENTS

### ee.csv preparation
### Observable Notebook functions/charts:
#### 1.) "plot_ee" / "chart_ee" (i.e., Significant earthquakes within 500 km since 1900)

In [134]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

df = pd.read_csv(files['earthquake_file'])

geometry = [Point(xy) for xy in zip(df['longitude'], df['latitude'])]
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs='EPSG:4326')

aoi = gpd.read_file(os.path.join( user_input_dir, 'AOI', city_name + '.shp')).to_crs('EPSG:4326')


# Use World Azimuthal Equidistant projection centered on AOI
aoi_centroid = aoi.to_crs('EPSG:4326').geometry.iloc[0].centroid
aeqd_proj = f'+proj=aeqd +lat_0={aoi_centroid.y} +lon_0={aoi_centroid.x} +x_0=0 +y_0=0 +datum=WGS84 +units=m'

gdf_proj = gdf.to_crs(aeqd_proj).copy()
aoi_proj = aoi.to_crs(aeqd_proj).copy()


# # Create 500km buffer and select points within it
buffer_500km = aoi_proj.buffer(500000)
nearby_points = gdf_proj[gdf_proj.within(buffer_500km.iloc[0])]

nearby_points.loc[:, 'distance_m'] = nearby_points.geometry.apply(
    lambda x: aoi_proj.geometry.distance(x).min()
)


nearby_points.loc[:, 'distance_km'] = nearby_points.distance(aoi_proj.geometry.iloc[0]) / 1000
nearby_points = nearby_points.to_crs('EPSG:4326')

nearby_points.columns



/Users/vivaldirinaldi/miniforge3/envs/databitch/lib/python3.10/site-packages/geopandas/geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
/Users/vivaldirinaldi/miniforge3/envs/databitch/lib/python3.10/site-packages/geopandas/geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


Index(['id', 'year', 'month', 'day', 'hour', 'minute', 'second',
       'locationName', 'latitude', 'longitude', 'eqMagnitude', 'eqMagUnk',
       'publish', 'country', 'regionCode', 'intensity', 'damageAmountOrder',
       'housesDamagedAmountOrder', 'damageAmountOrderTotal',
       'housesDamagedAmountOrderTotal', 'eqDepth', 'deathsAmountOrder',
       'housesDestroyedAmountOrder', 'tsunamiEventId', 'eqMagMw', 'eqMagMs',
       'deathsAmountOrderTotal', 'housesDestroyedAmountOrderTotal', 'deaths',
       'housesDestroyed', 'housesDamaged', 'deathsTotal',
       'housesDestroyedTotal', 'housesDamagedTotal', 'area', 'injuries',
       'injuriesAmountOrder', 'injuriesTotal', 'injuriesAmountOrderTotal',
       'volcanoEventId', 'eqMagMl', 'eqMagMb', 'damageMillionsDollars',
       'damageMillionsDollarsTotal', 'eqMagMfa', 'missingAmountOrder',
       'missingTotal', 'missingAmountOrderTotal', 'missing', 'geometry',
       'distance_m', 'distance_km'],
      dtype='object')

In [145]:
nearby_points = nearby_points[['year', 'month', 'day', 'eqMagnitude', 'distance_km', 'locationName', 'damageAmountOrder','deaths']]


nearby_points['BEGAN'] = pd.to_datetime(
    nearby_points[['year', 'month', 'day']], 
    errors='coerce'
).dt.strftime('%Y-%m-%d')


nearby_points['line1'] = pd.to_datetime(
    nearby_points[['year', 'month']].assign(day=1), 
    errors='coerce'
).dt.strftime('%B %Y').str.upper()

nearby_points['line2'] = nearby_points.apply(
    lambda row: f"M{row['eqMagnitude']}; {int(row['distance_km'])} km away" 
    if pd.notna(row['eqMagnitude']) 
    else f"MNA; {int(row['distance_km'])} km away",
    axis=1
)

damage_map = {
    1.0: 'Limited damage',
    2.0: 'Moderate damage',
    3.0: 'Severe damage',
    4.0: 'Extreme damage'
}
nearby_points['line3'] = nearby_points['damageAmountOrder'].map(damage_map).fillna('NA damage')



nearby_points['line4'] = nearby_points.apply(
    lambda row: f"{int(row['deaths'])} fatalities" 
    if pd.notna(row['deaths']) and row['deaths'] > 1
    else f"{int(row['deaths'])} fatality" 
    if pd.notna(row['deaths']) and row['deaths'] == 1
    else '',
    axis=1
)

# Create combined text column like in the first table
nearby_points['text'] = (
    nearby_points['line1'] + '; ' + 
    nearby_points['line2'] + '; ' + 
    nearby_points['line3'] + 
    nearby_points['line4'].apply(lambda x: '; ' + x if x else '')
)


nearby_points[['BEGAN', 'text', 'line1', 'line2', 'line3', 'line4', 'distance_km', 'eqMagnitude', 'locationName']].rename(columns={
    'distance_km': 'distance',
    'locationName': 'location'})

,BEGAN,text,line1,line2,line3,line4,distance,eqMagnitude,location
114,1906-09-17,SEPTEMBER 1906; MNA; 320 km away; NA damage,SEPTEMBER 1906,MNA; 320 km away,NA damage,,320.929521,NaN,ITALY: SICILY
125,1907-02-21,FEBRUARY 1907; MNA; 320 km away; NA damage,FEBRUARY 1907,MNA; 320 km away,NA damage,,320.929521,NaN,ITALY: SICILY
149,1908-12-28,DECEMBER 1908; M7.0; 483 km away; Extreme dama...,DECEMBER 1908,M7.0; 483 km away,Extreme damage,78000 fatalities,483.289886,7.0,"ITALY: MESSINA, SICILY, CALABRIA"
173,1909-10-21,OCTOBER 1909; MNA; 438 km away; NA damage,OCTOBER 1909,MNA; 438 km away,NA damage,,438.838018,NaN,ITALY: SICILY
207,1911-10-15,OCTOBER 1911; M4.3; 438 km away; NA damage,OCTOBER 1911,M4.3; 438 km away,NA damage,,438.838018,4.3,ITALY: ETNA
242,1914-05-08,MAY 1914; M4.9; 438 km away; Severe damage; 12...,MAY 1914,M4.9; 438 km away,Severe damage,120 fatalities,438.838018,4.9,"ITALY: CATANIA, ETNA"
394,1924-03-16,MARCH 1924; M5.6; 475 km away; NA damage,MARCH 1924,M5.6; 475 km away,NA damage,,475.368950,5.6,ALGERIA: BATNA
428,1926-08-17,AUGUST 1926; M5.3; 436 km away; Severe damage,AUGUST 1926,M5.3; 436 km away,Severe damage,,436.725386,5.3,ITALY: SALINA ISLAND
500,1930-03-26,MARCH 1930; MNA; 419 km away; Moderate damage,MARCH 1930,MNA; 419 km away,Moderate damage,,419.807207,NaN,ITALY: FILICUDI ISLAND
534,1931-07-06,JULY 1931; MNA; 430 km away; NA damage,JULY 1931,MNA; 430 km away,NA damage,,430.173348,NaN,ITALY: SICILY


In [ ]:
# EARTHQUAKE EVENTS - ee.csv preparation for Observable Notebook plot functions/charts:
# 1.) "plot_ee"/"chart_ee" (Significant Earthquakes within 500 km since 1900)

# load "raw" (i.e. "dirty") tabular output data
filename = get_file_by_topic('earthquake-events', tabular, tabular_dir)
raw_df_ee = pd.read_csv(filename) # updatefile path

# basic info about raw data
print("Raw earthquake events data info:")
print(f"Shape: {raw_df_ee.shape}")
print(f"Columns: {list(raw_df_ee.columns)}")
print(f"Total data points: {len(raw_df_ee)}")

# preview key data ranges if available
if 'eqMagnitude' in raw_df_ee.columns:
    print(f"Magnitude range: {raw_df_ee['eqMagnitude'].min():.1f} - {raw_df_ee['eqMagnitude'].max():.1f}")
if 'distance' in raw_df_ee.columns:
    print(f"Distance range: {raw_df_ee['distance'].min():.0f} - {raw_df_ee['distance'].max():.0f} km")

print(f"Data preview:")
print(raw_df_ee.head())
print("\n" + "="*50 + "\n")

# clean data using clean_ee function in clean.py
try:
    cleaned_df_ee = clean_ee(filename) # updatefile path
    print("Earthquake events data cleaned successfully!")
    
    # cleaned data info
    print(f"\nCleaned data shape: {cleaned_df_ee.shape}")
    print(f"Cleaned data columns: {list(cleaned_df_ee.columns)}")
    print(f"Sample of cleaned data:")
    print(cleaned_df_ee.head(10))
    
    # basic data validation
    print(f"\nData validation:")
    print(f"- Missing values: {cleaned_df_ee.isnull().sum().sum()}")
    print(f"- Year range: {cleaned_df_ee['begin_year'].min()} - {cleaned_df_ee['begin_year'].max()}")
    print(f"- Magnitude range: {cleaned_df_ee['eqMagnitude'].min():.1f} - {cleaned_df_ee['eqMagnitude'].max():.1f}")
    print(f"- Distance range: {cleaned_df_ee['distance'].min():.0f} - {cleaned_df_ee['distance'].max():.0f} km")
    
    # earthquake data analysis
    print(f"\nEarthquake Data Summary:")
    
    # basic statistics
    avg_magnitude = cleaned_df_ee['eqMagnitude'].mean()
    avg_distance = cleaned_df_ee['distance'].mean()
    
    print(f"- Total events: {len(cleaned_df_ee)}")
    print(f"- Average magnitude: {avg_magnitude:.1f}")
    print(f"- Average distance: {avg_distance:.0f} km")
    print(f"- Standard deviation (magnitude): {cleaned_df_ee['eqMagnitude'].std():.1f}")
    print(f"- Standard deviation (distance): {cleaned_df_ee['distance'].std():.0f} km")
    
    # temporal distribution
    years_span = cleaned_df_ee['begin_year'].max() - cleaned_df_ee['begin_year'].min()
    events_per_decade = (len(cleaned_df_ee) / years_span) * 10 if years_span > 0 else 0
    
    print(f"- Time span: {years_span} years")
    print(f"- Average events per decade: {events_per_decade:.1f}")
    
    # ID extremes
    strongest_eq = cleaned_df_ee.loc[cleaned_df_ee['eqMagnitude'].idxmax()]
    closest_eq = cleaned_df_ee.loc[cleaned_df_ee['distance'].idxmin()]
    
    print(f"- Strongest earthquake: {strongest_eq['eqMagnitude']:.1f} magnitude in {strongest_eq['begin_year']}")
    print(f"- Closest earthquake: {closest_eq['distance']:.0f} km in {closest_eq['begin_year']}")
    
    # data quality checks
    print(f"\nData Quality Checks:")
    
    quality_issues = 0
    
    # check for missing values in key columns
    missing_magnitude = cleaned_df_ee['eqMagnitude'].isna().sum()
    missing_distance = cleaned_df_ee['distance'].isna().sum()
    missing_year = cleaned_df_ee['begin_year'].isna().sum()
    
    if missing_magnitude > 0:
        print(f"Missing magnitude values: {missing_magnitude}")
        quality_issues += 1
    if missing_distance > 0:
        print(f"Missing distance values: {missing_distance}")
        quality_issues += 1
    if missing_year > 0:
        print(f"Missing year values: {missing_year}")
        quality_issues += 1
    
    # check for impossible values
    negative_magnitude = (cleaned_df_ee['eqMagnitude'] < 0).sum()
    negative_distance = (cleaned_df_ee['distance'] < 0).sum()
    
    if negative_magnitude > 0:
        print(f"Negative magnitude values: {negative_magnitude}")
        quality_issues += 1
    if negative_distance > 0:
        print(f"Negative distance values: {negative_distance}")
        quality_issues += 1
    
    # check for duplicate events (same year, magnitude, and distance)
    duplicates = cleaned_df_ee.duplicated(subset=['begin_year', 'eqMagnitude', 'distance']).sum()
    if duplicates > 0:
        print(f"Potential duplicate events: {duplicates}")
        quality_issues += 1
    
    if quality_issues == 0:
        print("No data quality issues detected")
        
except Exception as e:
    print(f"Error cleaning earthquake events data: {e}")
    print("Check that the earthquake-events.csv file exists and has the correct format")
    print("Expected columns: BEGAN, eqMagnitude, distance, text, line1, line2, line3")

# save confirmation and next steps
if 'cleaned_df_ee' in locals():
    print(f"\nCleaned data saved to: data/processed/ee.csv")
    
    # preview of data structure
    print(f"\nData structure summary:")
    print(f"- Columns: {list(cleaned_df_ee.columns)}")
    print(f"- Time series length: {len(cleaned_df_ee)} events")
    print(f"- Year range: {cleaned_df_ee['begin_year'].min()} - {cleaned_df_ee['begin_year'].max()}")
    print(f"- Data types: {dict(cleaned_df_ee.dtypes)}")
    
else:
    print("No cleaned earthquake data available")
    print("Troubleshooting steps:")
    print("   1. Ensure earthquake-events.csv exists in data/raw/")
    print("   2. Check file has correct columns: BEGAN, eqMagnitude, distance, text, line1, line2, line3")
    print("   3. Verify data contains parseable dates in BEGAN column")
    print("   4. Check that magnitude and distance values are numeric")

IndexError: list index out of range

### LIQUEFACTION SUSCEPTIBILITY

### l_area.csv preparation (i.e., % area with different liquefaction susceptibility levels - "No Data" (0); "Very low" (1); "Low" (2); "Medium" (3); "High" (4); and "Very high" (5))
### Observable Notebook functions/charts:
#### 1.) "plot_l_area" / "chart_l_area" (i.e., Percentage of area with different liquefaction susceptibiltiy levels, "No Data" (0); "Very low" (1); "Low" (2); "Medium" (3); "High" (4); and "Very high" (5)

In [53]:
# LIQUEFACTION SUSCEPTIBILITY - l_area.csv preparation from raw tif data for Observable Notebook plot functions/charts:
# 1.) "plot_l_area"/"chart_l_area" (i.e., Percentage of Area with different liquefaction susceptibility levels, "No Data" (0); "Very low" (1); "Low" (2); "Medium" (3); "High" (4); and "Very high" (5)

# load "raw" (i.e. "dirty") tif data

# input_tif_path = 'data/raw/2025-04-colombia-cartagena_02-process-output_spatial_cartagena_liquefaction.tif'

input_tif_path = get_file_by_topic('liquefaction.', raster, spatial_dir)

print("="*60)
print("LIQUEFACTION SUSCEPTIBILITY DATA PROCESSING")
print("="*60)

# data value distribution
print("Analyzing TIF data structure...")

try:
    import rasterio
    with rasterio.open(input_tif_path) as src:
        data = src.read(1)
        unique_vals = np.unique(data[~np.isnan(data)])
        print(f"Unique values in TIF: {unique_vals}")
        print(f"Data range: {data.min()} to {data.max()}")
        print(f"NoData value: {src.nodata}")
        
        # count pixels for each value
        for val in unique_vals:
            count = np.sum(data == val)
            print(f"Value {val}: {count:,} pixels")

except Exception as e:
    print(f"Could not examine TIF structure: {e}")

print("\n" + "-"*40)

# process tif file using clean_l_area function in clean.py
# set "include_nodata=False" to exclude value, "0" ("NoData"/background)
try:
    cleaned_df_liquefaction = clean_l_area(input_tif_path, include_nodata=False)
    print("Liquefaction susceptibility data processed successfully!")
    
    # cleaned data structure
    print(f"\nCleaned data shape: {cleaned_df_liquefaction.shape}")
    print(f"Cleaned data columns: {list(cleaned_df_liquefaction.columns)}")
    print(f"\nProcessed Liquefaction Susceptibility data:")
    print(cleaned_df_liquefaction)
    
    # basic data validation
    print(f"\nData Validation:")
    print(f"- Missing values: {cleaned_df_liquefaction.isnull().sum().sum()}")
    print(f"- Susceptibility categories: {len(cleaned_df_liquefaction)}")
    print(f"- Total pixels: {cleaned_df_liquefaction['count'].sum():,.0f}")
    print(f"- Percentage sum: {cleaned_df_liquefaction['percentage'].sum():.1f}% (should be ~100%)")
    
    # liquefaction susceptibility distribution analysis
    print(f"\nLiquefaction Susceptibility Distribution:")
    
    total_pixels = cleaned_df_liquefaction['count'].sum()
    
    for idx, row in cleaned_df_liquefaction.iterrows():
        if row['count'] > 0:  # only show categories with data
            print(f"- {row['bin']}: {row['count']:,.0f} pixels ({row['percentage']:.1f}%)")
    
    # risk level analysis
    if len(cleaned_df_liquefaction) > 0:
        # filter out zero-count categories
        active_categories = cleaned_df_liquefaction[cleaned_df_liquefaction['count'] > 0]
        
        if len(active_categories) > 0:
            max_susceptibility = active_categories.loc[active_categories['percentage'].idxmax()]
            min_susceptibility = active_categories.loc[active_categories['percentage'].idxmin()]
            
            print(f"\n- Most common susceptibility: {max_susceptibility['bin']} ({max_susceptibility['percentage']:.1f}%)")
            print(f"- Least common susceptibility: {min_susceptibility['bin']} ({min_susceptibility['percentage']:.1f}%)")
    
    # risk level groupings
    very_low_risk = cleaned_df_liquefaction[cleaned_df_liquefaction['bin'].isin(['Very low'])]['percentage'].sum()
    low_risk = cleaned_df_liquefaction[cleaned_df_liquefaction['bin'].isin(['Low'])]['percentage'].sum()
    medium_risk = cleaned_df_liquefaction[cleaned_df_liquefaction['bin'] == 'Medium']['percentage'].sum()
    high_risk = cleaned_df_liquefaction[cleaned_df_liquefaction['bin'].isin(['High'])]['percentage'].sum()
    very_high_risk = cleaned_df_liquefaction[cleaned_df_liquefaction['bin'].isin(['Very high'])]['percentage'].sum()

    
    print(f"\nRisk Level Summary:")
    print(f"- Very low risk areas: {very_low_risk:.1f}%")
    print(f"- Low risk areas: {low_risk:.1f}%")
    print(f"- Medium risk areas: {medium_risk:.1f}%")
    print(f"- High risk areas: {high_risk:.1f}%")
    print(f"- Very high risk areas: {very_high_risk:.1f}%")

    
    # data quality checks
    print(f"\nData Quality Checks:")
    
    quality_issues = 0
    
    # check for missing values
    missing_values = cleaned_df_liquefaction.isnull().sum().sum()
    if missing_values > 0:
        print(f"Missing values detected: {missing_values}")
        quality_issues += 1
    
    # check for negative values (should not exist)
    negative_counts = (cleaned_df_liquefaction['count'] < 0).sum()
    negative_percentages = (cleaned_df_liquefaction['percentage'] < 0).sum()
    
    if negative_counts > 0:
        print(f"Negative count values: {negative_counts}")
        quality_issues += 1
    if negative_percentages > 0:
        print(f"Negative percentage values: {negative_percentages}")
        quality_issues += 1
    
    # check percentage sum
    percentage_sum = cleaned_df_liquefaction['percentage'].sum()
    if abs(percentage_sum - 100) > 0.1:
        print(f"Percentage sum deviation: {percentage_sum:.1f}% (should be ~100%)")
        quality_issues += 1
    
    # check for duplicate categories
    duplicates = cleaned_df_liquefaction['bin'].duplicated().sum()
    if duplicates > 0:
        print(f"Duplicate susceptibility categories: {duplicates}")
        quality_issues += 1
    
    # check for expected liquefactionsusceptibility categories
    expected_categories = ['Very low', 'Low', 'Medium', 'High', 'Very high']
    actual_categories = cleaned_df_liquefaction['bin'].tolist()
    missing_categories = set(expected_categories) - set(actual_categories)
    if missing_categories:
        print(f"Categories not present in data: {missing_categories}")
    
    if quality_issues == 0:
        print("No data quality issues detected")
    
except Exception as e:
    print(f"Error processing liquefaction susceptibility data: {e}")
    print("Troubleshooting steps:")
    print("   1. Ensure TIF file exists at the specified path")
    print("   2. Check that the TIF file contains values 1-5 (or 0-5)")
    print("   3. Verify the TIF file is not corrupted")
    print("   4. Ensure rasterio library is installed: pip install rasterio")
    print("   5. Check file permissions and disk space")
    print("   6. If data includes value 0, try setting include_nodata=True")

print("\n" + "="*60)

# optional: to include value, "0" in analysis
# print("\n" + "="*30 + " INCLUDING VALUE 0 " + "="*30)
# try:
#     cleaned_df_with_zero = clean_liquefaction_area(input_tif_path, 
#                                                   output_file='data/processed/l_area_with_nodata.csv',
#                                                   include_nodata=True)
#     print("Alternative analysis (including value 0) completed!")
#     print(cleaned_df_with_zero)
# except Exception as e:
#     print(f"Error in alternative analysis: {e}")

LIQUEFACTION SUSCEPTIBILITY DATA PROCESSING
Analyzing TIF data structure...
Unique values in TIF: [  0   1   2   3   4   5 255]
Data range: 0 to 255
NoData value: 255.0
Value 0: 27 pixels
Value 1: 98 pixels
Value 2: 10 pixels
Value 3: 145 pixels
Value 4: 149 pixels
Value 5: 38 pixels
Value 255: 433 pixels

----------------------------------------
Unique values found in data: [1 2 3 4 5]
Cleaned liquefaction data saved to: data/processed/l_area.csv
Susceptibility categories: 5
Total pixels analyzed: 440
Percentage coverage verification: 100.0% (should be ~100%)
Dominant susceptibility level: High (33.9%)
Liquefaction susceptibility data processed successfully!

Cleaned data shape: (5, 4)
Cleaned data columns: ['bin', 'susceptibility', 'count', 'percentage']

Processed Liquefaction Susceptibility data:
         bin susceptibility  count  percentage
0   Very low              1     98       22.27
1        Low              2     10        2.27
2     Medium              3    145       32.95


### FIRE WEATHER INDEX (FWI)

### fwi.csv
### Observable Notebook functions/charts:
#### 1.) "plot_fwi" / "chart_fwi" (i.e., Fire Weather Index (FWI), January - December)

In [56]:
# FIRE WEATHER INDEX - fwi.csv preparation for Observable Notebook plot functions/charts:
# 1.) "plot_fwi"/"chart_fwi" (Fire Weather Index (FWI), January - December)

# load "raw" (i.e. "dirty") tabular output data

filename = get_file_by_topic('fwi', tabular, tabular_dir)
raw_df_fwi = pd.read_csv(filename) # updatefile path

# basic info about raw data
print("Raw fire weather index data info:")
print(f"Shape: {raw_df_fwi.shape}")
print(f"Columns: {list(raw_df_fwi.columns)}")
print(f"Total data points: {len(raw_df_fwi)}")

# preview key data ranges if available
if 'week' in raw_df_fwi.columns:
    print(f"Week range: {raw_df_fwi['week'].min()} - {raw_df_fwi['week'].max()}")
if 'pctile_95' in raw_df_fwi.columns:
    print(f"FWI range: {raw_df_fwi['pctile_95'].min():.2f} - {raw_df_fwi['pctile_95'].max():.2f}")

print(f"Data preview:")
print(raw_df_fwi.head())
print("\n" + "="*50 + "\n")

# clean data using clean_fwi function in clean.py
try:
    cleaned_df_fwi = clean_fwi(filename) # updatefile path
    print("Fire weather index data cleaned successfully!")
    
    # cleaned data info
    print(f"\nCleaned data shape: {cleaned_df_fwi.shape}")
    print(f"Cleaned data columns: {list(cleaned_df_fwi.columns)}")
    print(f"Sample of cleaned data:")
    print(cleaned_df_fwi.head(10))
    
    # basic data validation
    print(f"\nData validation:")
    print(f"- Missing values: {cleaned_df_fwi.isnull().sum().sum()}")
    print(f"- Week coverage: {len(cleaned_df_fwi)} weeks")
    print(f"- Week range: {cleaned_df_fwi['week'].min()} - {cleaned_df_fwi['week'].max()}")
    print(f"- FWI range: {cleaned_df_fwi['fwi'].min():.2f} - {cleaned_df_fwi['fwi'].max():.2f}")
    
    # fire weather analysis
    print(f"\nFire Weather Data Summary:")
    
    # basic statistics
    avg_fwi = cleaned_df_fwi['fwi'].mean()
    median_fwi = cleaned_df_fwi['fwi'].median()
    std_fwi = cleaned_df_fwi['fwi'].std()
    
    print(f"- Total weeks: {len(cleaned_df_fwi)}")
    print(f"- Average FWI: {avg_fwi:.2f}")
    print(f"- Median FWI: {median_fwi:.2f}")
    print(f"- Standard deviation: {std_fwi:.2f}")
    
    # ID extremes
    peak_week = cleaned_df_fwi.loc[cleaned_df_fwi['fwi'].idxmax()]
    lowest_week = cleaned_df_fwi.loc[cleaned_df_fwi['fwi'].idxmin()]
    
    print(f"- Highest FWI: {peak_week['fwi']:.2f} (Week {peak_week['week']}, {peak_week['monthName']}, {peak_week['danger']})")
    print(f"- Lowest FWI: {lowest_week['fwi']:.2f} (Week {lowest_week['week']}, {lowest_week['monthName']}, {lowest_week['danger']})")
    
    # danger level distribution
    danger_counts = cleaned_df_fwi['danger'].value_counts()
    print(f"\nDanger Level Distribution:")
    for level in ['Very low', 'Low', 'Moderate', 'High', 'Very high', 'Extreme']:
        count = danger_counts.get(level, 0)
        percentage = (count / len(cleaned_df_fwi)) * 100
        print(f"- {level}: {count} weeks ({percentage:.1f}%)")
    
    # monthly statistics
    monthly_stats = cleaned_df_fwi.groupby('monthName')['fwi'].agg(['mean', 'max', 'min']).round(2)
    
    print(f"\nMonthly FWI Statistics:")
    for month in ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']:
        if month in monthly_stats.index:
            stats = monthly_stats.loc[month]
            print(f"- {month}: Mean {stats['mean']:.2f}, Range {stats['min']:.2f} - {stats['max']:.2f}")
    
    # data quality checks
    print(f"\nData Quality Checks:")
    
    quality_issues = 0
    
    # check for missing values in key columns
    missing_week = cleaned_df_fwi['week'].isna().sum()
    missing_fwi = cleaned_df_fwi['fwi'].isna().sum()
    missing_month = cleaned_df_fwi['monthName'].isna().sum()
    missing_danger = cleaned_df_fwi['danger'].isna().sum()
    
    if missing_week > 0:
        print(f"Missing week values: {missing_week}")
        quality_issues += 1
    if missing_fwi > 0:
        print(f"Missing FWI values: {missing_fwi}")
        quality_issues += 1
    if missing_month > 0:
        print(f"Missing month values: {missing_month}")
        quality_issues += 1
    if missing_danger > 0:
        print(f"Missing danger values: {missing_danger}")
        quality_issues += 1
    
    # check for impossible values
    negative_fwi = (cleaned_df_fwi['fwi'] < 0).sum()
    if negative_fwi > 0:
        print(f"Negative FWI values: {negative_fwi}")
        quality_issues += 1
    
    # check for week sequence
    expected_weeks = set(range(1, 54))  # (Note: 53 weeks in a year) 
    actual_weeks = set(cleaned_df_fwi['week'].unique())
    missing_weeks = expected_weeks - actual_weeks
    if len(missing_weeks) > 0:
        print(f"Missing weeks: {sorted(list(missing_weeks))}")
        quality_issues += 1
    
    # check for duplicate weeks
    duplicates = cleaned_df_fwi['week'].duplicated().sum()
    if duplicates > 0:
        print(f"Duplicate week entries: {duplicates}")
        quality_issues += 1
    
    # check for valid danger categories
    valid_dangers = {'Very low', 'Low', 'Moderate', 'High', 'Very high', 'Extreme', 'Unknown'}
    invalid_dangers = set(cleaned_df_fwi['danger'].unique()) - valid_dangers
    if len(invalid_dangers) > 0:
        print(f"Invalid danger categories: {invalid_dangers}")
        quality_issues += 1
    
    if quality_issues == 0:
        print("No data quality issues detected")
        
except Exception as e:
    print(f"Error cleaning fire weather index data: {e}")
    print("Check that the FWI CSV file exists and has the correct format")
    print("Expected columns: week, pctile_95")

# save confirmation and next steps
if 'cleaned_df_fwi' in locals():
    print(f"\nCleaned data saved to: data/processed/fwi.csv")
    
    # preview of data structure
    print(f"\n Data structure summary:")
    print(f"- Columns: {list(cleaned_df_fwi.columns)}")
    print(f"- Time series length: {len(cleaned_df_fwi)} weeks")
    print(f"- Data types: {dict(cleaned_df_fwi.dtypes)}")
    print(f"- FWI value range: {cleaned_df_fwi['fwi'].min():.2f} - {cleaned_df_fwi['fwi'].max():.2f}")
    
else:
    print("No cleaned fire weather data available")
    print("Troubleshooting steps:")
    print("   1. Ensure FWI CSV file exists in data/raw/")
    print("   2. Check file has correct columns: week, pctile_95")
    print("   3. Verify week numbers are sequential (1-53)")
    print("   4. Check that FWI values are numeric and non-negative")

Raw fire weather index data info:
Shape: (53, 2)
Columns: ['week', 'pctile_95']
Total data points: 53
Week range: 1 - 53
FWI range: 28.35 - 95.59
Data preview:
   week  pctile_95
0     1  36.038556
1     2  28.351868
2     3  34.486138
3     4  35.160120
4     5  41.215546


Cleaned data saved to: data/processed/fwi.csv
Weeks covered: 53 weeks
Week range: 1 - 53
FWI range: 28.35 - 95.59
Danger level distribution:
  Very low: 0 weeks (0.0%)
  Low: 0 weeks (0.0%)
  Moderate: 0 weeks (0.0%)
  High: 15 weeks (28.3%)
  Very high: 13 weeks (24.5%)
  Extreme: 25 weeks (47.2%)
Peak fire weather month: Jul (max FWI: 95.59)
Fire weather index data cleaned successfully!

Cleaned data shape: (53, 4)
Cleaned data columns: ['week', 'monthName', 'fwi', 'danger']
Sample of cleaned data:
   week monthName    fwi     danger
0     1       Jan  36.04       High
1     2       Jan  28.35       High
2     3       Jan  34.49       High
3     4       Jan  35.16       High
4     5       Feb  41.22  Very high
5 